# [실습] 계약별 요율표 — 계수를 배율로 읽는다

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 오늘의 목표와 데이터

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">모형추정 심화통계 · Day 3</div>
<h1 style="color:#000000;margin:10px 0">일반화선형모형 — 계수 하나가 요율 문장이 된다</h1>
<div style="font-size:1.05em;color:#656565">어제의 242개 세그먼트 요율표를 계약 단위 요율로 전환합니다</div>
</div>

## 오늘 할 일

어제 우리는 계약 678,013건 전체의 연간 사고율을 0.1006 건/년 한 숫자로 확정했습니다. 그 숫자를 242개 **세그먼트**(지역·브랜드 같은 조건의 조합 하나와, 그 조합에 속한 계약 전부)로 나누고, 관측이 적은 세그먼트는 사전분포로 보정한 것까지가 어제입니다. 오늘 언더라이팅 팀에서 답장이 왔습니다 — **"세그먼트별 평균은 알겠습니다. 그런데 같은 세그먼트 안에서도 사람이 다른데, 계약마다 다른 요율을 어떻게 매기나요?"**

그래서 오늘은 세그먼트를 더 쪼개는 대신 **계약의 특성을 입력으로 받아 그 계약만의 사고율을 계산하는 회귀 모형**을 만듭니다. 이론에서 배운 GLM 의 구성요소 3개와 오프셋을 그대로 실제 계약 데이터에 적용합니다. 계수를 **요율 문장**으로 옮기고, 후보 여럿을 같은 조건에서 비교해 선택하는 단계까지 진행합니다. 요율 문장이란 요율표에 그대로 적을 수 있는 「무엇이 1단위 오르면 사고율이 몇 배」 형태의 한 문장을 말합니다.

수업에서 함께 도는 것은 미션 1·2 와 정리이고, 미션 3 과 과제는 방과후에 진행합니다. 뒤로 미뤄도 정리를 먼저 읽으면 결론이 이어집니다.

## 오늘의 데이터

**freMTPL2freq** 는 프랑스 자동차 책임보험(Motor Third-Party Liability)의 실제 계약 기록입니다. 계약 한 건이 한 행이고 678,013행 × 12컬럼입니다. 어제와 같은 데이터·같은 전처리이고, 공개 저장소 OpenML 에서 로그인 없이 내려받습니다.

| 컬럼 | 뜻 | 단위 | 받자마자 고정하는 규칙 |
|---|---|---|---|
| `ClaimNb` | 보험 기간 중 사고 건수 | 건 | 상한 4건으로 상한 처리 — 사고 5건 이상은 극소수이고 대부분 입력 오류·특수 사례입니다 |
| `Exposure` | 그 계약이 실제로 유지된 기간, 곧 **노출** | 년 | 상한 1.0년으로 상한 처리 — 1년을 초과하는 노출은 기록 관례의 잔재입니다 |
| `DrivAge` | 운전자 나이 | 살 | 그대로 씁니다 |
| `BonusMalus` | **할증점수** — 프랑스 자동차보험의 할인·할증 등급 | 점 | 그대로 씁니다. 기준 100점, 무사고 해마다 내려가 최저 50점, 사고를 내면 올라가는 50～230점 범위의 지수입니다(이론 2. GLM 의 구성요소 3개에서 쓴 점수 범위와 같고, 미션 1 의 구간 경계도 230 에서 끝납니다) |

이 규칙 2개는 5일 내내 고정합니다. 그 위에서 나온 전체 계약(회사 전체)의 사고율 0.1006 건/년을 오늘 모든 비교의 기준값으로 씁니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_tweedie_regression_insurance_claims_001.png" width="640" height="320"/>
<p align="center"><em>▲ 오늘의 목적지 — 왼쪽 위 train data 그래프에서 구간별 관측 사고빈도(파랑 점)가 18세 0.30 에서 30세 0.06 근처까지 감소해 평평해지고, 모형 예측(주황 점)이 그 곡선과 거의 일치한다 (출처: scikit-learn documentation)</em></p>

나머지 그래프들은 같은 예제의 test data 쪽이고 가로축과 세로축 범위가 서로 다르니, 오늘은 왼쪽 위 한 장만 봅니다. 오늘 만들 모형이 하는 일이 저 주황 점을 산출하는 것이고, 읽을 것은 「젊은 구간만 높았다가 30세 뒤로 평평해진다」는 모양 하나입니다.

## 이 노트북을 쓰는 법

**환경 준비** — ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」 설치 뒤 사용합니다(온보딩 레슨 참고). **채점은 Colab 이 연결되면 자동으로 준비됩니다 — 코드 셀 머리의 「채점 준비됨」이 뜬 뒤 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 안 찍히면 그 「채점 준비됨」을 누르세요.**

노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다). AI 튜터 대신 다른 AI 에 PROMPT 를 붙여 넣어 받은 코드를 `# --- 여기부터 AI 코드 ---` 시작 행 아래에 직접 붙입니다.

**맨 위 설치·폰트 셀을 먼저 실행하세요 — 한 번이면 됩니다.** 미션마다 코드 셀이 2개입니다. 앞의 것은 **준비 셀**(그대로 실행), 뒤의 것은 **PROMPT 셀**(시킬 일을 적는 곳)입니다. `# --- 여기부터 AI 코드 ---` 이 **AI 코드 시작 행**이고, 그 아래가 AI 가 코드를 작성하는 위치입니다.

실행하는 순간 **실행 결과(출력)에 채점 결과**가 출력됩니다. 기준을 충족하면 완성본 풀이가 열리고, 못 미치면 모자란 항목과 AI 튜터에 보낼 질문이 나옵니다. 변수 이름은 무엇으로 지어도 됩니다. 채점 기준은 이름이 아니라 셀이 출력한 값과 그림이기 때문입니다.

며칠 뒤 뒤쪽 미션만 열어도 맨 위 설치·폰트 셀 다음에 그 미션의 「준비 셀」부터 실행하면 됩니다.

**주고받기 규칙 2가지** — 막혔을 때는 이 둘만 지킵니다.

| 이럴 때 | 이렇게 합니다 |
|---|---|
| 같은 말을 두 번째 되풀이하게 될 때 | 멈춥니다. 말을 바꾸지 말고 `PROMPT` 에 **빠진 조건을 하나 더합니다** |
| 그래도 안 될 때 | 앞 대화를 이어서 쓰지 말고, 요구를 한 덩어리로 다시 적어 새 메시지로 보냅니다 |

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "scipy>=1.11" "matplotlib>=3.7" "statsmodels>=0.14" "scikit-learn>=1.3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150      # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 모형·강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다

print("사용 폰트:", FONT, "| 색 팔레트 COL 과 캔버스 폭 FIG_W 준비 완료")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 바로 아래에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 □ 로 표시되지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 □ 로 표시될 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 튜플로 복원합니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6eyJ2YWx1ZXMiOnsi7Jew6rCEIOyCrOqzoOycqCAo6rG0L+uFhCkiOlswLjEw"
           "MDEsMC4xMDExLCLqsbQv64WEIl0sIm51bGwg66qo7ZiV7J2YIOuhnOq3uOyasOuPhCI6"
           "Wy0xNDY3MDAuMCwtMTQ2NTYwLjAsIiJdfSwicGxvdCI6eyJraW5kIjoibGluZSIsIm5f"
           "c2VyaWVzIjoxLCJ4X3JhbmdlIjpbNTIuNSwxOTAuMF0sInBlYWtfeCI6WzE1MC4wLDIz"
           "MC4wXSwiaGxpbmVfeSI6WzAuMDk5NSwwLjEwMTddLCJ5c2NhbGUiOiJsb2ciLCLstpXr"
           "nbzrsqgiOnRydWV9fSwibTIiOnsidmFsdWVzIjp7Iu2VoOymneygkOyImCAx7KCQ7J2Y"
           "IOyCrOqzoOycqOu5hCAo67CwKSI6WzEuMDIxNSwxLjAyMjYsIuuwsCJdLCLrj4Tsi5zt"
           "mZQg65Ox6riJIEYg7J2YIOyCrOqzoOycqOu5hCAo67CwKSI6WzEuNDUyLDEuNDYyLCLr"
           "sLAiXX0sInBsb3QiOnsia2luZCI6ImJhcmgiLCJuIjo4LCJ4bWF4IjpbMS40NCwxLjQ4"
           "XSwidmxpbmVfeCI6WzAuOTksMS4wMV0sInhsYWJlbCI6dHJ1ZX19LCJtMyI6eyJ2YWx1"
           "ZXMiOnsi6rCA7J6lIOuCruydgCBBSUMgKOygkCkiOlsyODcyNzIuMCwyODcyNzQuMCwi"
           "7KCQIl0sIuyxhOygkO2RnCBudWxsIO2WieydmCDroZzqt7jsmrDrj4QiOlstMTQ2NjQw"
           "LjAsLTE0NjYxMC4wLCIiXX0sInBsb3QiOnsia2luZCI6ImJhciIsIm4iOjMsInltYXgi"
           "OlsyOTI4MDAuMCwyOTM3MDAuMF0sInltaW4iOlsyODcyNTAuMCwyODczMDAuMF0sImFy"
           "Z21heCI6Im51bGwiLCLstpXrnbzrsqgiOnRydWV9fSwi6rO87KCcIjp7InZhbHVlcyI6"
           "eyLssKjrn4kg66qo7ZiV7J2YIEFJQyAo7KCQKSI6WzI5MTI4MS4wLDI5MTI4NC4wLCLs"
           "oJAiXSwi7LCo65+JIOuqqO2YleydmCDssKjroLkgMeuFhCDsgqzqs6DsnKjruYQgKOuw"
           "sCkiOlswLjk2NTIsMC45NjYzLCLrsLAiXSwi6rKs67O4IOqzhOyVvSDshLgg6rG0IOyY"
           "iOy4oSDsmpTsnKjsnZgg7LWc64yAwrfstZzshowg67mEICjrsLApIjpbMi4xMDg1LDIu"
           "MTE4LCLrsLAiXX0sInBsb3QiOnsia2luZCI6ImxpbmUiLCJuX3NlcmllcyI6MiwieF9y"
           "YW5nZSI6WzAuMCwxMDAuMF0sInBlYWtfeCI6WzgwLjAsMTAwLjBdLCJ5c2NhbGUiOiJs"
           "b2ciLCLstpXrnbzrsqgiOnRydWV9fX0="
)
_REVEAL = {"m1":                   # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
           "8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIEFJIOy9lOuTnCDsi5zsnpEg7ZaJIO"
           "yVhOuemOyXkCDrk6TslrTsmYDslbwg7ZWgIOy9lOuTnOuKlCDsnbQg7KCV64+E7J6F"
           "64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpbgg7ZSE66Gs7ZSE7Yq4IOyYiA"
           "oKICAgIOydtOuvuCDrp4zrk6TslrTsoLgg7J6I64qUIGRmLCBvZmYsIENMQUlNU19B"
           "TEwsIEVYUF9BTEwsIGJtX3gsIGJtX3JhdGUg66W8IOq3uOuMgOuhnCDsk7Dqs6Ag64"
           "uk7IucIOunjOuTpOyngOuKlCDrp5DslYQg7KSYLiBzdGF0c21vZGVscyDroZwg7ISk"
           "66qF67OA7IiYCiAgICDsl4bsnbQg7KCI7Y6466eMIOyeiOuKlCDtj6zslYTshqEg7Z"
           "qM6reA66W8IGRmIOyXkCDsoIHtlantlbQg7KSYIOKAlCDsooXsho3rs4DsiJjripQg"
           "Q2xhaW1OYiDsnbTqs6Ag64W47LacIOuztOygleydgCBvZmZzZXQg7JeQIG9mZiDrpb"
           "wg64SY6riw66m0IOuPvC4g7KCB7ZWp7ZWcIOuSpCDsoIjtjrjsnYQg66Gc6re4IOyK"
           "pOy8gOydvCDqt7jrjIDroZwKICAgIO2VnCDtloksIOq3uCDsoIjtjrjsnYQg7KeA7I"
           "iY66W8IOy3qO2VtCDrs7Xsm5DtlZwg6rCS7J2EICLsl7DqsIQg7IKs6rOg7JyoID0g"
           "KOyIq+yekCkg6rG0L+uFhCIg7LKY65+8IOyGjOyImOygkCDslYTrnpggNuyekOumrO"
           "q5jOyngCDri6jsnITsmYAg7ZWo6ruYIO2VnCDtloksIOq3uCDrqqjtmJXsnZgg66Gc"
           "6re47Jqw64+E66W8ICLroZzqt7jsmrDrj4QgPSAo7Iir7J6QKSIg66GcCiAgICDtlZ"
           "wg7ZaJIHByaW50IO2VtCDspJguIOq3uOuemO2UhOuKlCDrlLEg7ZWcIOyepeunjCDi"
           "gJQg6rCA66Gc7LaV7JeQIGJtX3go7ZWg7Kad7KCQ7IiYIOq1rOqwhCDspJHslZnqsJ"
           "IpLCDshLjroZzstpXsl5AgYm1fcmF0ZSjqtazqsITrs4Qg6rSA7LihIOyXsOqwhCDs"
           "gqzqs6DsnKgp66W8IOuGk+qzoCDsoJDsnYQg7J207J2AIOqzoeyEoOydhAogICAg7Z"
           "WY64KY66eMIOq3uOumsCDrkqQsIOuwqeq4iCDqtaztlZwg7Jew6rCEIOyCrOqzoOyc"
           "qCDqsJLsl5Ag66ee7LawIOqwgOuhnOyEoOydhCDtlZjrgpgg6riL6rOgIOyEuOuhnO"
           "y2leyXkOuKlCDroZzqt7gg64iI6riI7J2EIOyNqCDspJguIOuRkCDstpXsl5Ag6528"
           "67Ko7J2EIOuLrOqzoCDqs6HshKDqs7wg6rCA66Gc7ISg7J20IOqwgeqwgSDrrLTsl4"
           "fsnbjsp4Ag67KU66GA66GcIOq1rOu2hO2VtCDspJguCgogICAgcmVzX251bGwgPSBz"
           "bWYuZ2xtKGZvcm11bGE9IkNsYWltTmIgfiAxIiwgZGF0YT1kZiwKICAgICAgICAgIC"
           "AgICAgICAgICAgICBmYW1pbHk9c20uZmFtaWxpZXMuUG9pc3NvbigpLCBvZmZzZXQ9"
           "b2ZmKS5maXQoKQogICAgbGFtX251bGwgPSBmbG9hdChucC5leHAocmVzX251bGwucG"
           "FyYW1zLmlsb2NbMF0pKQoKICAgIHByaW50KGYi66Gc6re4IOyKpOy8gOydvCDsoIjt"
           "jrggOiB7cmVzX251bGwucGFyYW1zLmlsb2NbMF06LjZmfSIpCiAgICBwcmludChmIu"
           "yXsOqwhCDsgqzqs6DsnKggICAgICA6IHtsYW1fbnVsbDouNmZ9IOqxtC/rhYQiKQog"
           "ICAgcHJpbnQoZiLroZzqt7jsmrDrj4QgICAgICAgICA6IHtyZXNfbnVsbC5sbGY6LC"
           "4yZn0iKQoKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oRklHX1cs"
           "IDQpKQogICAgYXgucGxvdChibV94LCBibV9yYXRlLCAiby0iLCBjb2xvcj1DT0xbIu"
           "2VmOuKmOyDiSJdLCBsdz0yLCBsYWJlbD0i6rWs6rCE67OEIOq0gOy4oSDsgqzqs6Ds"
           "nKgiKQogICAgYXguYXhobGluZShsYW1fbnVsbCwgY29sb3I9Q09MWyLrgqjsg4kiXS"
           "wgbHM9IjoiLCBsYWJlbD0ibnVsbCDrqqjtmJXsnbQg66ek6riw64qUIOuLqOydvCDs"
           "mpTsnKgiKQogICAgYXguc2V0X3lzY2FsZSgibG9nIikgICAgICAgICAgICAgICAgIC"
           "AgICAgICAgICAgICAgICAgIyDroZzqt7gg64iI6riI7J2065287JW8IOuCruydgCDq"
           "tazqsITsnZgg6riw7Jq46riw6rCAIOuztOyeheuLiOuLpAogICAgYXguc2V0X3hsYW"
           "JlbCgi7ZWg7Kad7KCQ7IiYICjqtazqsIQg7KSR7JWZ6rCSKSIpCiAgICBheC5zZXRf"
           "eWxhYmVsKCLsl7DqsIQg7IKs6rOg7JyoICjqsbQv64WELCDroZzqt7gg64iI6riIKS"
           "IpCiAgICBheC5sZWdlbmQoKQogICAgcGx0LnNob3coKQoK7ZW07ISdCgotIOygiO2O"
           "uOydgCDroZzqt7gg7Iqk7LyA7J287J2YIOqwkuydtOudvCAtMi4yOTY0NjUg66GcIO"
           "y2nOugpeuQqeuLiOuLpC4g7KeA7IiY66W8IOyUjOybjOyVvCAwLjEwMDYxNCDqsbQv"
           "64WEIOKAlCDslrTsoJwg7IaQ7Jy866GcIO2RvCDqsJLqs7wg7IaM7IiY7KCQIOyVhO"
           "uemCDsl6zshK/sp7gg7J6Q66as6rmM7KeAIOqwmeydgCDqsJLsnoXri4jri6QuIOyY"
           "iOy4oSDsp4jrrLjsl5DshJwgIuygiO2OuOydgCDsnYzsiJjsnbjrjbAg7KeA7IiY66"
           "W8IOyUjOyasOuptCAwLjEwMDYiIOydhCDqs6DrpbTshajri6TrqbQg66ee7JWY6rOg"
           "LCAi7KCI7Y647J20IOq3uOuMgOuhnCAwLjEwMDYiIOydhCDqs6DrpbTshajri6Trqb"
           "Qg7J20IOywqOydtOqwgCDroZzqt7gg66eB7YGsIOqzhOyImOydmCDsiqTsvIDsnbzs"
           "nYQg7ZmV7J247ZWY64qUIOyngOygkOyeheuLiOuLpC4g66Gc6re4IOunge2BrCDrqq"
           "jtmJXsnZgg6rOE7IiY64qUIOyghOu2gCDroZzqt7gg7Iqk7LyA7J287J2YIOqwkuyd"
           "tOudvCDqt7jrjIDroZwg7J297Jy866m0IOydjOyImOqwgCDrgpjsmLXri4jri6QuCi"
           "0g7J20IOqwkuyXkCDstJ0g64W47LacIDM1OCwzNjAuMSDrhYTsnYQg6rOx7ZWY66m0"
           "IDM2LDA1NiDqsbQsIOyLpOygnCDsgqzqs6Ag6rG07IiY7JmAIOyGjOyImOygkCDslY"
           "TrnpjquYzsp4Ag7J287LmY7ZWp64uI64ukLiDsoIjtjrjrp4wg65GUIO2PrOyVhOyG"
           "oSBHTE0g7J2YIOy1nOuMgOyasOuPhO2VtOqwgCDqsrDqta0gIuy0nSDsgqzqs6Agw7"
           "cg7LSdIOuFuOy2nCIg7J20652864qUIOqyg+ydhCDrqqjtmJXsnbQg6rCZ7J2AIOqy"
           "sOqzvOuhnCDri6Tsi5wg7IKw7Lac7ZWcIOqyg+yeheuLiOuLpC4KLSDroZzqt7jsmr"
           "Drj4QgLTE0Niw2MjQuMzgg7J2AIOyWtOygnCDstZzshoztmZTtlZwg7J2M7J2YIOuh"
           "nOq3uOyasOuPhCAxNDYsNjI0IOyZgCDrtoDtmLjrp4wg64uk66W4IOqwmeydgCDqsJ"
           "LsnoXri4jri6QuIEdMTSDsoIHtlansnYAg7IOIIOybkOumrOqwgCDslYTri4jrnbwg"
           "7Ja07KCc7J2YIOy1nOuMgOyasOuPhOy2lOygleydhCDqs4TsiJgg7Jes65+sIOqwnO"
           "uhnCDrhJPtnowg6rKD7J206rOgLCDsmKTripgg66eM65OkIO2bhOuztCDrqqjtmJXs"
           "nYAg7KCE67aAIOydtCDqsJIo6rOE7IiYIDHqsJzsnbgg6riw7KSA7ISgKeqzvCDruY"
           "TqtZDtlanri4jri6QuCi0gZmFtaWx5IOulvCDsoIHsp4Ag7JWK7Jy866m0IHN0YXRz"
           "bW9kZWxzIOuKlCDqsr3qs6Ag7JeG7J20IOygleq3nOu2hO2PrMK37ZWt65OxIOunge"
           "2BrOuhnCDsoIHtlantlanri4jri6QuIOq3uOufrOuptCDsmIjsuKEg7IKs6rOgIOqx"
           "tOyImOqwgCDsnYzsiJjsnbgg6rOE7JW97J20IDM3JSDrgpjsmKTqs6AsIGV4cCDroZ"
           "wg67OA7ZmY7ZWcIOqwkuuPhCDrnLvsnbQg7JeG7Ja07KeR64uI64ukIOKAlCDqsJLq"
           "s7wg6re4656Y7ZSE6rCAIO2VqOq7mCDti4Drpqzqsowg6rOE7IKw65CY64qUIOqyve"
           "yasOyeheuLiOuLpC4KLSDqt7jrprzsnYAg7ZWcIOusuOyepeycvOuhnCDsmpTslb3r"
           "kKnri4jri6QuIOq0gOy4oSDsgqzqs6DsnKjsnYAg7ZWg7Kad7KCQ7IiY66W8IOuUsO"
           "udvCAwLjA3OTgg7JeQ7IScIDAuNTY5MiDquYzsp4AgNy4x67CwIOyYpOultOuKlOuN"
           "sCwgbnVsbCDrqqjtmJXsnbQg6re47J2AIOqyg+ydgCAwLjEwMDYg7JeQIOuGk+yduC"
           "DqsIDroZzshKAg7ZWY64KY7J6F64uI64ukLiAxNTDsoJAg7J207IOBIOq1rOqwhOyX"
           "kOyEnOuKlCDqtIDsuKHsnbQg66qo7ZiV7J2YIDUuN+uwsOydtOqzoCwgNTB+NTXsoJ"
           "Ag6rWs6rCE7JeQ7ISc64qUIOyYpO2eiOugpCDrqqjtmJXsnbQg64aS7Iq164uI64uk"
           "LiDri6Trp4wg6re4IOunqCDsmKTrpbjsqr0g6rWs6rCE7J2AIOqzhOyVvSAyMDjqsb"
           "Qgwrcg64W47LacIDk0LjnrhYQgwrcg7IKs6rOgIDU06rG067+Q7J6F64uI64ukIOKA"
           "lCDtkZzspIDsmKTssKjqsIAg4oiaNTQgw7cgOTQuOSDiiYggMC4wNzcg6rG0L+uFhC"
           "wg6rCS7J2YIDE0JSDrnbwgNS4367Cw652864qUIOqwkiDsnpDssrTqsIAgNeuwsOyZ"
           "gCA267CwIOyCrOydtOyXkOyEnCDrtojtmZXsi6Ttlanri4jri6QuIOuwqe2WpSjsmK"
           "Trpbjsqr3snLzroZwg6rCI7IiY66GdIOychO2XmOydtCDrhpLri6Qp7J2AIOuvv+qz"
           "oCwg7J6Q66a/7IiY64qUIOuLpOydjCDrr7jshZjsnZgg66qo7ZiV7JeQIOunoeq5ge"
           "uLiOuLpC4g7J20IOywqOydtOulvCDri6TsnYwg66+47IWY7JeQ7IScIOqzhOyImCDt"
           "lZjrgpjtlZjrgpjqsIAg7ISk66qF7ZWY6rKMIOuQqeuLiOuLpC4KCuy9lOuTnOqwgC"
           "Dri7XsnYQg66eM65Oc64qUIOuwqeyLnSDigJQg7Jqw67OA7JeQIOygiO2OuCDtlZjr"
           "gpjrp4wg65GUIO2PrOyVhOyGoSBHTE0g7JeQIGxvZyhFeHBvc3VyZSkg66W8IOqzhO"
           "yImCAxIOuhnCDqs6DsoJXtlbQg642U7ZWY66m0LCDstZzrjIDsmrDrj4TtlbTqsIAg"
           "IuyCrOqzoCDqsbTsiJjsnZgg7ZWpIMO3IOuFuOy2nOydmCDtlakiIO2VnCDsiKvsnp"
           "DroZwg65ao7Ja07KeR64uI64ukLiDqt7gg6rCS7J2AIOuhnOq3uCDsiqTsvIDsnbzs"
           "l5Ag7J6I7Jy864uIIGV4cCDroZwg67OA7ZmY7ZW07JW8IOqxtC/rhYQg64uo7JyE7J"
           "2YIOqwkuydtCDrkKnri4jri6QuCg==",
           "m2":
           "8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIOqzoOyzkOyVvCDtlaAg6rOz7J2AIO"
           "OAjOuzteybkO2VmOuKlCDsvZTrk5wg65GQIO2WieOAjSDrv5DsnoXri4jri6QKCuKW"
           "oCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7KCE7J"
           "6EIOuLtOuLueyekCDsvZTrk5zsl5DshJwg7KCB7ZWpIO2WiShyZXNfY29yZSnqs7wg"
           "7KCI7Y647J2EIOu6gCDqs4TsiJggOOqwnOulvCDri7TsnYAgYmV0YSDtlonsnYAg6r"
           "e464yA66GcIOuRkOqzoCwg6re4IOqzhOyImOulvCDsl63rs4DtmZjtlZjripQg7L2U"
           "65OcIOuRkCDtlonrp4wg6rOg7LOQIOykmC4g7Y+s7JWE7IahIEdMTSDsnZgKICAgIO"
           "qzhOyImOuKlCDroZzqt7gg7Iqk7LyA7J287J2YIOqwkuydtOudvCAxMDAg7J2EIOqz"
           "se2VtOyEnOuKlCDrsLDsnKjsnbQg65CY7KeAIOyViuyVhCDigJQg6rOE7IiY7JeQIO"
           "yngOyImOulvCDslIzsmrQg6rCS7J20IOyCrOqzoOycqOu5hCByciDsnbTqs6AsIOuw"
           "seu2hOycqCDrs4DtmZQgcGN0IOuKlCDqt7ggcnIg7JeQ7IScIDEg7J2EIOu6gCDrkq"
           "QgMTAwIOydhAogICAg6rOx7ZWcIOqwkuydtOyVvC4g6rOg7LmcIOuSpCAi7ZWg7Kad"
           "7KCQ7IiYIDHsoJDsnZgg7IKs6rOg7Jyo67mEID0gKOyIq+yekCkg67CwIiDsmYAgIu"
           "uPhOyLnO2ZlCDrk7HquIkgRiDsnZgg7IKs6rOg7Jyo67mEID0gKOyIq+yekCkg67Cw"
           "IiDrkZAg6rCS7J2EIOyGjOyImOygkCDslYTrnpggNuyekOumrOq5jOyngCDri6jsnI"
           "TsmYAg7ZWo6ruYIOy2nOugpSDtlZwg7ZaJ7JSpCiAgICBwcmludCDtlbQg7KSYIOKA"
           "lCDqs4TsiJgg7J2066aE7J2AIEJvbnVzTWFsdXMg7JmAIEMoQXJlYSlbVC5GXSDslb"
           "wuIOq3uOumvOydgCDsoITsnoTsnpAg7L2U65Oc7JeQIOyXhuycvOuLiCDsg4jroZwg"
           "6re466CkIOykmCDigJQg6rOE7IiYIDjqsJzsnZgg7IKs6rOg7Jyo67mE66W8IOqwgO"
           "uhnCDrp4nrjIAgOOqwnOuhnAogICAg6re466as6rOgLCDjgIzsmIHtlqUg7JeG7J2M"
           "44CN7JeQIO2VtOuLue2VmOuKlCAxLjAg7JeQIOyEuOuhnCDsoJDshKDsnYQg7ZWY64"
           "KYIOq4i+qzoCwg6rCA66Gc7LaV7JeQIOustOyXh+ydmCDrsLDsnKjsnbjsp4Ag6528"
           "67Ko7J2EIOuLrOyVhCDspJguCgogICAgYmV0YSA9IHJlc19jb3JlLnBhcmFtcy5kcm"
           "9wKCJJbnRlcmNlcHQiKSAgICMg7KCB7ZWpIOy9lOuTnCDtlonqs7wg7J20IO2Wieyd"
           "gCDqt7jrjIDroZwg65Gh64uI64ukCiAgICByciA9IG5wLmV4cChiZXRhKSAgICAgIC"
           "AgICAgICAgICAgICAgICAgICAgIyDroZzqt7gg7Iqk7LyA7J287J2YIOqzhOyImOul"
           "vCDsm5Drnpgg64uo7JyE7J2YIOuwsOycqOuhnAogICAgcGN0ID0gKHJyIC0gMSkgKi"
           "AxMDAgICAgICAgICAgICAgICAgICAgICAgICMg67Cw7Jyo7JeQ7IScIDEg7J2EIOu5"
           "vOqzoCAxMDAg7J2EIOqzse2VmOuptCDrsLHrtoTsnKgg67OA7ZmUCgogICAgcHJpbn"
           "QoZiLtlaDspp3soJDsiJggMeygkOydmCDsgqzqs6DsnKjruYQgOiB7ZmxvYXQocnJb"
           "J0JvbnVzTWFsdXMnXSk6LjZmfSDrsLAiKQogICAgcHJpbnQoZiLrj4Tsi5ztmZQg65"
           "Ox6riJIEYg7J2YIOyCrOqzoOycqOu5hCA6IHtmbG9hdChyclsnQyhBcmVhKVtULkZd"
           "J10pOi42Zn0g67CwIikKCuq3uOumvOydgCDtkZzrpbwg6rOg7LmcIOuSpOyXkCDqsJ"
           "nsnYAg7ZSE66Gs7ZSE7Yq47JeQ7IScIO2VqOq7mCDsi5ztgrXri4jri6QuCgogICAg"
           "c2hvdyA9IFsiQyhBcmVhKVtULkJdIiwgIkMoQXJlYSlbVC5DXSIsICJDKEFyZWEpW1"
           "QuRF0iLCAiQyhBcmVhKVtULkVdIiwgIkMoQXJlYSlbVC5GXSIsCiAgICAgICAgICAg"
           "ICJDKFZlaEdhcylbVC5SZWd1bGFyXSIsICJEcml2QWdlIiwgIkJvbnVzTWFsdXMiXQ"
           "ogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPShGSUdfVywgNC40KSkK"
           "ICAgIGJhcnMgPSBheC5iYXJoKHNob3csIHJyW3Nob3ddLCBjb2xvcj1DT0xbIuyXsO"
           "2ajOyDiSJdKQogICAgYmFyc1stMV0uc2V0X2NvbG9yKENPTFsi7ZWY64qY7IOJIl0p"
           "ICAgICAgICAgICAgICAgICAgICAjIOyYpOuKmCDsmpTsnKgg66y47J6l7Jy866GcIO"
           "yYruq4uCDtla3rqqnrp4wg6rCV7KGwCiAgICBmb3IgYiwgdiBpbiB6aXAoYmFycywg"
           "cnJbc2hvd10pOgogICAgICAgIGF4LnRleHQodiArIDAuMDA2LCBiLmdldF95KCkgKy"
           "BiLmdldF9oZWlnaHQoKSAvIDIsIGYie3Y6LjRmfSIsIHZhPSJjZW50ZXIiKQogICAg"
           "YXguYXh2bGluZSgxLjAsIGNvbG9yPUNPTFsi64Ko7IOJIl0sIGxzPSI6IikgICAgIC"
           "AgICAgICMg44CM7JiB7ZalIOyXhuydjOOAjSDquLDspIAKICAgIGF4LnNldF94bGlt"
           "KDAuOTcsIDEuNTMpCiAgICBheC5zZXRfeGxhYmVsKCLsgqzqs6DsnKjruYQg4oCUIO"
           "yCrOqzoOycqOyXkCDqs7HtlbTsp4DripQg67Cw7JyoIikKICAgIHBsdC5zaG93KCkK"
           "Cu2VtOyEnQoKLSDsoIHtlakg7L2U65OcIO2WieydgCDshpDrjIgg642w6rCAIOyXhu"
           "yXiOyKteuLiOuLpC4g6rSA7LihIDY3OCwwMTMg6rG0IOq3uOuMgOuhnOydtOqzoCDs"
           "mKTtlITshYvrj4Qg65Ok7Ja06rCU7Jy866mwLCDsmIjsuKEg7IKs6rOgIOqxtOyImO"
           "ydmCDtlakgMzYsMDU2IOqxtOydtCDsi6TsoIHqs7wg7KCV7ZmV7Z6IIOydvOy5mO2V"
           "qeuLiOuLpCDigJQg7KCI7Y647J20IOyeiOuKlCDtj6zslYTshqEgR0xNIOydgCDsoI"
           "TssrQg7ZWp7J2EIOunnuy2lOqyjCDrkJjslrQg7J6I7Iq164uI64ukLiDti4DrprAg"
           "6rOz7J2AIOqzhOyImOulvCDrsLDsnKjroZwg67OA7ZmY7ZWY64qUIO2WiSDtlZjrgp"
           "jrv5DsnbTsl4jsirXri4jri6QuCi0g7ZWo7KCVIOy9lOuTnOqwgCDsk7QgMSArIGJl"
           "dGEg64qUIOyngOyImO2VqOyImOydmCAx7LCoIOq3vOyCrOyeheuLiOuLpC4g7ZWg7K"
           "ad7KCQ7IiY7JeQ7ISc64qUIDEuMDIxODE2IOuMgCAxLjAyMjA1NiDsnbTrnbwg7LGE"
           "7KCQ7J20IOq3uOuMgOuhnCDthrXqs7ztlojsp4Drp4wsIOuPhOyLnO2ZlCDrk7HquI"
           "kgRiDsl5DshJzripQgMS4zNzYxNzYg64yAIDEuNDU2NzA0IOuhnCDssKjsnbTqsIAg"
           "7Luk7KC4IOqygOy2nOuQmOyXiOyKteuLiOuLpC4g6rCSIO2VmOuCmOqwgCDthrXqs7"
           "ztlojri6Tqs6Ag7L2U65OcIOyghOyytOqwgCDrp57ripQg6rKD7J2AIOyVhOuLiOud"
           "vOuKlCDsmIjsnbTqs6AsIOyalOycqO2RnOyXkCDsi6TrpqzripQg6rKD7J2AIOuMgO"
           "qwnCDqs4TsiJjqsIAg7YGwIO2VreuqqeydtOudvCDrjZQg6re466CH7Iq164uI64uk"
           "LgotIOyInOyEnOulvCDrsJTqvrjripQg6rKD64+EIO2dlO2VnCDsmKTrpZjsnoXri4"
           "jri6QuIOyngOyImOulvCDrqLzsoIAg7JSM7Jqw6rOgIDEg7J2EIOuCmOykkeyXkCDr"
           "uo3ri4jri6Qg4oCUIG5wLmV4cChiZXRhKSAtIDEg7J20IOunnuqzoCwg6rSE7Zi4IO"
           "yViOyXkOyEnCDrqLzsoIAg67m866m0IOyZhOyghO2eiCDri6Trpbgg6rCS7J20IOuC"
           "mOyYteuLiOuLpC4KLSDsmpTsnKgg66y47J6l7Jy866GcIOyYruq4sOuptCDsnbTroI"
           "fsirXri4jri6QuIO2VoOymneygkOyImOuKlCAx7KCQ64u5IDEuMDIyMDU2IOuwsCwg"
           "7KaJIDIuMjElIOyDgeyKueydtOqzoCDsoJDsiJjripQgMeygkOyUqSDspp3qsIDtlZ"
           "jsp4Ag7JWK7Jy866+A66GcIOqxsOuTreygnOqzseycvOuhnCDriITsoIHrkKnri4jr"
           "i6Qg4oCUIDIw7KCQIOywqOydtOuptCAxLjAyMjA1NiDsnZggMjDsoJzqs7EgPSAxLj"
           "U0NywgNTUlIO2VoOymneyeheuLiOuLpC4g64+E7Iuc7ZmUIOuTseq4ieydgCBBIOul"
           "vCDquLDspIDsnLzroZwgQiAxLjA3NTAgwrcgQyAxLjEyOTEgwrcgRCAxLjIzNTEgwr"
           "cgRSAxLjMzNjkgwrcgRiAxLjQ1Njcg67Cw66GcIOuIhOyggeuQqeuLiOuLpC4g67CY"
           "64yA66GcIEMoVmVoR2FzKVtULlJlZ3VsYXJdIOuKlCDrsLDsnKggMC45OTY2IOyXkC"
           "BwLeqwkiAwLjc1IOKAlCDtnJjrsJzsnKDsmYAg6rK97JygIOyCrOydtOyXkCDsgqzq"
           "s6DsnKgg7LCo7J206rCAIOyeiOuLpOqzoCDrp5DtlaAg6re86rGw6rCAIOydtCDrjb"
           "DsnbTthLDsl5DripQg7JeG64uk64qUIOucu+ydtOudvCwg6re466a87JeQ7IScIOyd"
           "tCDrp4nrjIDrp4wg7IS466Gc7ISgIOyZvOyqveyXkCDslYTso7wg7KGw6riIIOqxuO"
           "y5qeuLiOuLpC4KLSDsmIjsuKEg7KeI66y47J2YIOuLteydgCDslpHsiJjsnoXri4jr"
           "i6QuIERyaXZBZ2Ug7J2YIOuztOygleuQnCDqs4TsiJjripQgKzAuMDA3MTcg66GcIO"
           "uwsOycqCAxLjAwNzIg7J24642wLCDrgpjsnbTrp4wg7Zi87J6QIOuEo+qzoCDsoIHt"
           "lantlZjrqbQgLTAuMDA0Njcg7JeQIOuwsOycqCAwLjk5NTMg7Jy866GcIOu2gO2YuO"
           "qwgCDrsJjrjIDqsIAg65Cp64uI64ukLiAxOH4yNeyEuCDqtazqsITsnZgg6rSA7Lih"
           "IOyCrOqzoOycqOydtCAwLjE3NSDqsbQv64WE7Jy866GcIOqwgOyepSDrhpLsnYDrjb"
           "Ag6re4IOq1rOqwhOydmCDtj4nqt6Ag7ZWg7Kad7KCQ7IiY64+EIDg5LjAg7Jy866Gc"
           "IOqwgOyepSDrhpLqs6AsIOuRkCDrs4DsiJjsnZgg7IOB6rSA7J20IC0wLjQ4IOyehe"
           "uLiOuLpC4g7KCK7J2AIOyatOyghOyekOqwgCDsnITtl5jtlZwg7J207Jyg7J2YIOyD"
           "geuLuSDrtoDrtoTsnbQg7J2066+4IO2VoOymneygkOyImOyXkCDquLDroZ3rj7wg7J"
           "6I64uk64qUIOucu+yeheuLiOuLpC4g6re4656Y7IScIOuRkCDqs4TsiJjripQg7ISc"
           "66GcIOuLpOuluCDsp4jrrLjsl5Ag64u17ZWp64uI64ukIOKAlCDri6jrj4Ug6rOE7I"
           "iY64qUICLrgpjsnbTrp4wg7JWMIOuVjCIsIOuztOygleuQnCDqs4TsiJjripQgIu2V"
           "oOymneygkOyImOqwgCDqsJnsnYAg65GQIOyCrOuejCDsgqzsnbTsl5DshJwiIOuCmO"
           "ydtCAx7IK07J2YIOywqOydtOulvCDsuKHsoJXtlanri4jri6QuIO2VoOymneygkOyI"
           "mOulvCDsmpTsnKjtkZzsl5Ag7J2066+4IOuEo+yXiOuLpOuptCDrs7TsoJXrkJwg6r"
           "OE7IiY6rCAIOunnuuKlCDqsJLsnbTqs6AsIOyXrOq4sOyEnCDqt5zsnKjsnbQg7ZWY"
           "64KYIOuCmOyYteuLiOuLpDogR0xNIOqzhOyImOuKlCDslrjsoJzrgpgg44CM64uk66"
           "W4IOuzgOyImOulvCDqs6DsoJXtlojsnYQg65WM44CNIOydmCDtmqjqs7zsnbTrr4Dr"
           "oZwsIOuztOqzoOyEnOyXkCDqs4TsiJjrpbwg7KCB7J2EIOuVjOuKlCDrqqjtmJXsl5"
           "Ag66y07JeH7J20IO2VqOq7mCDrk6TslrQg7J6I7JeI64qU7KeA66W8IOuKmCDtlajq"
           "u5gg7KCB7Iq164uI64ukLgoKUFJPTVBUIOuKlCDsnbTrn7Ag66qo7JaR7J20IOuQqe"
           "uLiOuLpCDigJQg6rOg7LmgIO2WieydhCDsp4DrqqntlZjqs6AsIOyGkOuMgOyngCDr"
           "p5Ag67KU7JyE66W8IOu2hOuqhe2eiCDsoIHqs6AsIOyDiOuhnCDsi5ztgqwg6re466"
           "a87J2YIOyalOq1rOulvCDsoIHripQgM+qwnOqwgCDri6Qg65Ok7Ja0IOyeiOyKteuL"
           "iOuLpC4KCiAgICDrqqntkZwg4oCUIEFJIOy9lOuTnCDsi5zsnpEg7ZaJIOyVhOuemC"
           "DsvZTrk5zsl5DshJwg6rOE7IiY66W8IOuwsOycqOuhnCDsl63rs4DtmZjtlZjripQg"
           "7ZaJ7J20IO2LgOuguOuLpC4g66Gc6re4IOyKpOy8gOydvOydmCDqs4TsiJjrpbwKIC"
           "AgICAgICAgICDsm5Drnpgg64uo7JyE7J2YIOuwsOycqOuhnCDrs7Xsm5DtlZjqs6As"
           "IOuwseu2hOycqCDrs4DtmZTripQg6re4IOuwsOycqOyXkOyEnCDrp4zrk6TslrTrnb"
           "wuIOygge2VqSDsvZTrk5wg7ZaJ6rO8IOyduOyHhCDsvZTrk5wKICAgICAgICAgICDt"
           "lonsnYAg6re464yA66GcIOuRlOuLpC4KICAgIOyTuCDrs4DsiJgg4oCUIGRmIMK3IE"
           "ZfQ09SRSDCtyBvZmYgwrcgQ0xBSU1TX0FMTCwg6re466as6rOgIEFJIOy9lOuTnCDs"
           "i5zsnpEg7ZaJIOyVhOuemOyXkCDsnbTrr7gg7J6I64qUIHJlc19jb3JlIMK3IGJldG"
           "EuCiAgICDstpzroKUg4oCUIO2VoOymneygkOyImCAx7KCQ7J2YIOyCrOqzoOycqOu5"
           "hOyZgCDrj4Tsi5ztmZQg65Ox6riJIEYg7J2YIOyCrOqzoOycqOu5hOulvCDsnbTrpo"
           "Tqs7wg64uo7JyEKOuwsCnrpbwg67aZ7JesCiAgICAgICAgICAg7IaM7IiY7KCQIOyV"
           "hOuemCA27J6Q66as66GcIOy2nOugpSDtlZwg7ZaJ7JSpIOyduOyHhO2VnOuLpC4g6r"
           "e466as6rOgIOygiO2OuOydhCDruoAg6rOE7IiYIDjqsJzrpbwg7IS466Gc7LaV7JeQ"
           "LAogICAgICAgICAgIOyCrOqzoOycqOu5hOulvCDqsIDroZzstpXsl5Ag64aT7J2AIO"
           "qwgOuhnCDrp4nrjIAg7ZWcIOyepSDigJQg67Cw7JyoIDEuMCDsl5Ag7IS466Gc7ISg"
           "LCDqsIDroZzstpUKICAgICAgICAgICDrnbzrsqgg7Y+s7ZWoLgogICAg6rOE7ZqN7J"
           "2EIO2VnCDrrLjsnqXsnLzroZwg66i87KCAIOunkO2VmOqzoCwg6rOg7LmgIOy9lOuT"
           "nCDtlokg66eQ6rOg64qUIOq3uOuMgOuhnCDrkawuCgoi64uk7IucIOynnCDspJgiIO"
           "udvOqzoCDtlZjrqbQg66mA7Kmh7ZWcIOygge2Vqeq5jOyngCDtlajqu5gg67CU64CM"
           "7Ja0IOuPjOyVhOyYteuLiOuLpC4g64+M7JWE6rCA64qUIOy9lOuTnOulvCDqs6Dsua"
           "Ag65WM64qUIOqzoOy5oCDtlonqs7wg7IaQ64yA7KeAIOunkCDrspTsnITrpbwg7ZWo"
           "6ruYIOyggeuKlCDqsoPsnbQg7JqU66C57J6F64uI64ukLgoK7L2U65Oc6rCAIOuLte"
           "ydhCDrp4zrk5zripQg67Cp7IudIOKAlCDsoIHtlansnYAg66Gc6re4IOyKpOy8gOyd"
           "vOyXkOyEnCDqs4TsiJjrpbwg642U7ZW0IOuhnOq3uCDsgqzqs6DsnKjsnYQg66eM65"
           "Ok6rOgLCBleHAg6rCAIOq3uCDrjafshYjsnYQg7JuQ656YIOuLqOychOydmCDqs7Hs"
           "hYjsnLzroZwg67OA7ZmY7ZWp64uI64ukLiDqt7jrnpjshJwg6rOE7IiYIO2VmOuCmO"
           "2VmOuCmOqwgCDsgqzqs6DsnKjsl5Ag6rOx7ZW07KeA64qUIOuwsOycqCwg6rOnIOya"
           "lOycqCDrrLjsnqXsnbQg65Cp64uI64ukLgo=",
           "m3":
           "8J+UkyDrr7jshZggMyDsmYTshLHrs7gg4oCUIEFJIOy9lOuTnCDsi5zsnpEg7ZaJIO"
           "yVhOuemOyXkCDrk6TslrTsmYDslbwg7ZWgIOy9lOuTnOuKlCDsnbQg7KCV64+E7J6F"
           "64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpbgg7ZSE66Gs7ZSE7Yq4IOyYiA"
           "oKICAgIOydtOuvuCDrp4zrk6TslrTsoLgg7J6I64qUIGRmLCBGT1JNVUxBUywgRl9D"
           "T1JFLCBGX0ZVTEwsIG9mZiDrpbwg6re464yA66GcIOyTsOqzoCDri6Tsi5wg66eM65"
           "Ok7KeA64qUIOunkOyVhCDspJguIEZPUk1VTEFTIOyXkCDrk6Ag7ZuE67O0IDPqsJwo"
           "bnVsbCDCtwogICAgY29yZSDCtyBmdWxsKeydhCDrsJjrs7XrrLjsnLzroZwg7ZWY64"
           "KY7JSpIO2PrOyVhOyGoSBHTE0g7JeQIOygge2Vqe2VtCDspJgg4oCUIOuFuOy2nCDr"
           "s7TsoJXsnYAgb2Zmc2V0IOyXkCBvZmYg66W8IOuEmOq4sOuptCDrj7wuIOygge2Vqe"
           "2VoCDrlYzrp4jri6Qg66qo7ZiVIOydtOumhCwg7KCI7Y647J2EIO2PrO2VqO2VnCDq"
           "s4TsiJgKICAgIOqwnOyImCwg66Gc6re47Jqw64+ELCDsnbTtg4jrj4QsIEFJQywgQk"
           "lDLCDqtIDsuKEg7IiY66W8IO2VnCDtlonsnLzroZwg66qo7JWEIO2RnOuhnCDrp4zr"
           "k6Tqs6AgQUlDIOqwgCDrgq7snYAg7Iic7Jy866GcIOygleugrO2VtCDsnbjsh4Ttlb"
           "Qg7KSYIOKAlCBBSUMg64qUIOuhnOq3uOyasOuPhOyXkCAtMiDrpbwg6rOx7ZWcIOqw"
           "kuyXkAogICAg6rOE7IiYIOqwnOyImOydmCAy67Cw66W8IOuNlO2VnCDqsJIsIEJJQy"
           "DripQg7Yyo64SQ7YuwIO2VreydtCBsb2co6rSA7LihIOyImCnrsLDsnbgg6rKD66eM"
           "IOuLrOudvC4g7ZGc66W8IOywjeydgCDrkqQgIuqwgOyepSDrgq7snYAgQUlDID0gKO"
           "yIq+yekCkg7KCQIiDsnYQg7Ja064qQIOuqqO2YleyduOyngOyZgCDtlajqu5gg7Lac"
           "66ClIO2VnCDtloksCiAgICDssYTsoJDtkZwgbnVsbCDtlonsnZgg66Gc6re47Jqw64"
           "+E66W8IOy2nOugpSDtlZwg7ZaJIOuNlCBwcmludCDtlZjqs6AsIOyEuCDsoIHtlans"
           "nZgg6rSA7LihIOyImOqwgCDrqqjrkZAg6rCZ7J2A7KeA64+EIOy2nOugpSDtlZwg7Z"
           "aJ7Jy866GcCiAgICDtmZXsnbjtlbQg7KSYLiDqt7jrnpjtlITripQg65SxIO2VnCDs"
           "nqXrp4wg4oCUIOqwgOuhnOy2leyXkCBudWxsIMK3IGNvcmUgwrcKICAgIGZ1bGwg7I"
           "S4IOydtOumhOydhCDqt7gg7Iic7ISc64yA66GcIOuGk+qzoCDshLjroZzstpXsl5Ag"
           "6rCBIOuqqO2YleydmCBBSUMg66W8IOyEuOuhnCDrp4nrjIAgM+qwnOuhnCDqt7jrpr"
           "Ag65KkLCDqsIDsnqUg64Ku7J2AIOunieuMgCDtlZjrgpjrp4wg7IOJ7J2EIOuLrOum"
           "rO2VmOqzoCDrp4nrjIAg7JyE7JeQIOqwkuydhCDsoIHqs6Ag65GQIOy2leyXkCDrnb"
           "zrsqjsnYQKICAgIOuLrOyVhCDspJguCgogICAgcm93cyA9IFtdCiAgICBmb3IgbmFt"
           "ZSwgZm9ybXVsYSBpbiBGT1JNVUxBUy5pdGVtcygpOiAgICAgICAgICAgICAgICAgIC"
           "AjIOyEuCDtm4Trs7Trpbwg7ZWcIOuwmOuzteusuOyXkOyEnCDsoIHtlantlanri4jr"
           "i6QKICAgICAgICByID0gc21mLmdsbShmb3JtdWxhPWZvcm11bGEsIGRhdGE9ZGYsIG"
           "ZhbWlseT1zbS5mYW1pbGllcy5Qb2lzc29uKCksIG9mZnNldD1vZmYpLmZpdCgpCiAg"
           "ICAgICAgayA9IGludChyLmRmX21vZGVsKSArIDEgICAgICAgICAgICAgICAgICAgIC"
           "AgICAjIGRmX21vZGVsIOydgCDsoIjtjrjsnYQg67qAIOqwnOyImOudvCAxIOydhCDr"
           "jZTtlanri4jri6QKICAgICAgICBuID0gaW50KHIubm9icykKICAgICAgICByb3dzLm"
           "FwcGVuZCh7Im1vZGVsIjogbmFtZSwgImsiOiBrLCAibG9nbGlrIjogci5sbGYsICJk"
           "ZXZpYW5jZSI6IHIuZGV2aWFuY2UsCiAgICAgICAgICAgICAgICAgICAgICJhaWMiOi"
           "AtMiAqIHIubGxmICsgMiAqIGssCiAgICAgICAgICAgICAgICAgICAgICJiaWMiOiAt"
           "MiAqIHIubGxmICsgbnAubG9nKG4pICogaywgIm5vYnMiOiBufSkKCiAgICBzY29yZV"
           "90YWJsZSA9IHBkLkRhdGFGcmFtZShyb3dzKS5zb3J0X3ZhbHVlcygiYWljIikucmVz"
           "ZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcHJpbnQoc2NvcmVfdGFibGUucm91bmQoMi"
           "kudG9fc3RyaW5nKGluZGV4PUZhbHNlKSkKICAgIHByaW50KGYi6rCA7J6lIOuCruyd"
           "gCBBSUMgOiB7c2NvcmVfdGFibGVbJ2FpYyddLm1pbigpOiwuMmZ9IOygkCIKICAgIC"
           "AgICAgIGYiICAoe3Njb3JlX3RhYmxlLmxvY1tzY29yZV90YWJsZVsnYWljJ10uaWR4"
           "bWluKCksICdtb2RlbCddfSDrqqjtmJUpIikKICAgIHByaW50KGYi7LGE7KCQ7ZGcIG"
           "51bGwg7ZaJ7J2YIOuhnOq3uOyasOuPhCA9IHtmbG9hdChzY29yZV90YWJsZS5sb2Nb"
           "c2NvcmVfdGFibGVbJ21vZGVsJ10gPT0gJ251bGwnLCAnbG9nbGlrJ10uaWxvY1swXS"
           "k6LC4yZn0iKQogICAgcHJpbnQoIuyEuCDsoIHtlansnZgg6rSA7LihIOyImOqwgCDr"
           "qqjrkZAg6rCZ7J2A6rCAIDoiLCBzY29yZV90YWJsZVsibm9icyJdLm51bmlxdWUoKS"
           "A9PSAxKQoKICAgIG9yZGVyID0gWyJudWxsIiwgImNvcmUiLCAiZnVsbCJdCiAgICB2"
           "YWxzID0gW2Zsb2F0KHNjb3JlX3RhYmxlLmxvY1tzY29yZV90YWJsZVsibW9kZWwiXS"
           "A9PSBtLCAiYWljIl0uaWxvY1swXSkgZm9yIG0gaW4gb3JkZXJdCiAgICBmaWcsIGF4"
           "ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDgsIDQpKQogICAgYmFycyA9IGF4LmJhci"
           "hvcmRlciwgdmFscywgY29sb3I9ImxpZ2h0Z3JheSIpCiAgICBiYXJzW2ludChucC5h"
           "cmdtaW4odmFscykpXS5zZXRfY29sb3IoInRhYjpibHVlIikgICAjIOqwgOyepSDrgq"
           "7snYAg66eJ64yAIO2VmOuCmOunjCDqsJXsobAKICAgIGZvciBiLCB2IGluIHppcChi"
           "YXJzLCB2YWxzKToKICAgICAgICBheC50ZXh0KGIuZ2V0X3goKSArIGIuZ2V0X3dpZH"
           "RoKCkgLyAyLCB2ICsgMjAwLCBmInt2OiwuMGZ9IiwgaGE9ImNlbnRlciIpCiAgICBh"
           "eC5zZXRfeWxpbSgyODUwMDAsIDI5NTAwMCkKICAgIGF4LnNldF94bGFiZWwoIu2bhO"
           "uztCDrqqjtmJUiKQogICAgYXguc2V0X3lsYWJlbCgiQUlDICjsoJAsIOuCruydhOyI"
           "mOuhnSDsoovri6QpIikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSBBSUMg7Iic7I"
           "Sc64qUIGZ1bGwgMjg3LDI3Mi44MyA8IGNvcmUgMjg4LDE0OS44MSA8IG51bGwgMjkz"
           "LDI1MC43NiDsnoXri4jri6QuIO2ZleyepSDrqqjtmJXsnZggQUlDIOqwgCDtlbXsi6"
           "wg66qo7ZiV67O064ukIDg3N+ygkCDrgq7sirXri4jri6Qg4oCUIOyXrOufrOu2hCDq"
           "t7jrprzsl5DshJwg6rCV7KGw65CcIOqwgOyepSDrgq7snYAg66eJ64yA6rCAIOq3uO"
           "qyg+yeheuLiOuLpC4KLSBCSUMg66GcIOyInOychOulvCDrp6Tqsqjrj4Qg7Iic7ISc"
           "64qUIOqwmeyKteuLiOuLpCDigJQgZnVsbCAyODcsNzQxLjMzIDwgY29yZSAyODgsMj"
           "UyLjY1IDwgbnVsbCAyOTMsMjYyLjE4LiDsmIjsuKEg7KeI66y47J2YIOuztOq4sCA0"
           "6rCcIOykkSDri7XsnYAg7LKrIOuyiOynuCwg44CM65GYIOuLpCBmdWxsIOydtCAx65"
           "Ox44CNIOyeheuLiOuLpC4g7Yyo64SQ7YuwIO2VreydtCA267CwIO2BsOuNsOuPhCDs"
           "iJzsnITqsIAg7JWIIOuwlOuAkCDsnbTsnKDripQg67qE7IWI7J2EIO2VtCDrs7Trqb"
           "Qg67O07J6F64uI64ukLiDtmZXsnqUg66qo7ZiV7J2AIOqzhOyImOulvCAzMuqwnCDr"
           "jZQg7I2o7IScIOuhnOq3uOyasOuPhOulvCA0NzAuNDkg7Kad6rCA7Iuc7Lyw7Jy864"
           "uIIOKIkjLihJMg6riw7KSA7Jy866GcIDk0MC45OOygkOydhCDrsozsl4jqs6AsIOqx"
           "sOq4sOyEnCBBSUMg64qUIDIgeCAzMiA9IDY07KCQ7J2ELCBCSUMg64qUIDEzLjQyNj"
           "kgeCAzMiA9IDQyOS42NuygkOydhCDrl43ri4jri6QuIOuCqOuKlCDqsoPsnbQg6rCB"
           "6rCBIDg3N+ygkOqzvCA1MTHsoJDsnoXri4jri6QuIO2RnOuzuOydtCDtgazrqbQg67"
           "O17J6h7ZWcIOuqqO2YleydtCDsnKDrpqztlbTsp5Hri4jri6Qg4oCUIOydtOqyg+yd"
           "gCBBSUPCt0JJQyDsnZgg7ISx7KeI7J207KeAIO2ZleyepSDrqqjtmJXsnbQg7KeE66"
           "as652864qUIOucu+ydtCDslYTri5nri4jri6QuCi0gbnVsbCDtlonsnZgg66Gc6re4"
           "7Jqw64+E64qUIC0xNDYsNjI0LjM4IOuhnCDrr7jshZggMSDsl5DshJwg7Lac66Cl7Z"
           "WcIOq4sOykgOyEoCDqsJIg6re464yA66Gc7J6F64uI64ukLiDsnbQg7ZWcIO2Wieyd"
           "tCDjgIzshLgg7KCB7ZWp7J20IOqwmeydgCDqsoPsnYQg7Lih7KCV7ZWY6rOgIOyeiO"
           "uLpOOAjSDripQg7Kad6rGw7J6F64uI64ukIOKAlCDsmKTtlITshYvsnbQg67mg7KeA"
           "6rGw64KYIO2DgOq5g+ydtCDrsJTrgIzrqbQg7Jes6riw67aA7YSwIOqwkuydtCDrp5"
           "7sp4Ag7JWK7Iq164uI64ukLgotIOyxhOygkO2RnCAz6rCcIO2WieydmCDqtIDsuKEg"
           "7IiY64qUIOuqqOuRkCA2NzgsMDEzIOycvOuhnCDqsJnsirXri4jri6QuIOydtCDqsJ"
           "LsnbQg64us65287KeA66m0IOuhnOq3uOyasOuPhOyXkCDrjZTtlaAg7ZWt7J2YIOyI"
           "mOu2gO2EsCDri6zrnbzsoLgg7KCQ7IiY66W8IOu5hOq1kO2VoCDsiJgg7JeG7Iq164"
           "uI64ukLiDqtIDsuKEg7KeR7ZWp7J20IOyYpOulmCDsl4bsnbQg64us65287KeA64qU"
           "IOqwgOyepSDtnZTtlZwg6rK966Gc6rCAIOq1rOqwhO2ZlOuCmCDqsrDsuKEg7KCc6r"
           "Gw66GcIO2WieydtCDruaDsp4DripQg6rKD7J206528LCDtkZzsl5Ag7J20IOyXtOyd"
           "hCDrgqjqsqgg65GQ66m0IOq3uOumvCDsl4bsnbQg7ZmV7J247ZWgIOyImCDsnojsir"
           "Xri4jri6QuCi0g7J207YOI64+EKGRldmlhbmNlIOKAlCDsmYTrsr3tlZjqsowg66ee"
           "7Z6I64qUIOqwgOyDgeydmCDrqqjtmJXqs7wg64K0IOuqqO2YlSDsgqzsnbTsnZgg66"
           "Gc6re47Jqw64+EIOqyqeywqOulvCAy67CwIO2VnCDqsJIp64qUIG51bGwgMjIzLDkz"
           "MiAtPiBjb3JlIDIxOCw4MTUgLT4gZnVsbCAyMTcsODc0IOyeheuLiOuLpC4g7Yyo64"
           "SQ7YuwIO2VreydtCDsl4bslrTshJwg67OA7IiY66W8IOuEo+ydhOyImOuhnSDrrLTs"
           "obDqsbQg6rCQ7IaM7ZWp64uI64ukLiDsnbTtg4jrj4Trp4wg67O06rOgIOqzoOultO"
           "uptCDripgg6rCA7J6lIOuzteyeoe2VnCDtm4Trs7TqsIAg7ISg7YOd65Cp64uI64uk"
           "IOKAlCDssYTsoJDtkZzsl5Ag7J20IOyXtOydhCDtlajqu5gg7Iuk7Ja0IOuRlCDsnb"
           "TsnKDsnoXri4jri6QuCi0g65287J2067iM65+s66asIOq4sOuzuCDsho3shLHsnYQg"
           "7ZmV7J24IOyXhuydtCDsk7DrqbQg7YuA66as64qUIOqyveyasOqwgCDtlZjrgpgg7J"
           "6I7Iq164uI64ukLiBzdGF0c21vZGVscyDsnZggR0xNIOyXkOuKlCBiaWMg7IaN7ISx"
           "7J20IOuUsOuhnCDsnojripTrjbAsIOq3uOqyg+ydgCDsnbTtg4jrj4Qg6riw67CY7J"
           "2YIOuLpOuluCDsoJXsnZjrnbwg7Jqw66asIOyLneqzvCDqsJLsnbQg7KCE67aAIOuL"
           "pOumheuLiOuLpC4g67aA7Zi46rmM7KeAIOuwmOuMgOqwgCDrkJjslrQg7ZmV7J6lIO"
           "uqqO2YleyXkOyEnOuKlCDiiJI4LDg4NSwyMDMsIOOAjOydtOuhoCA1LiBBSUMg7JmA"
           "IEJJQ+OAjeqwgCDsoIHslrQg65GUIO2VteyLrCDrqqjtmJXsl5DshJzripQg4oiSOC"
           "w4ODQsNjkxLjMg7LKY65+8IOuCmOyYteuLiOuLpC4g66Gc6re47Jqw64+EIOq4sOuw"
           "mOydtCDtlYTsmpTtlZjrqbQgYmljX2xsZiDrpbwg66qF7Iuc7ZWY6rGw64KYLCDsnI"
           "Tsspjrn7wg7KCV7J2Y64yA66GcIOyngeygkSDslIHri4jri6QuCgrsvZTrk5zqsIAg"
           "64u17J2EIOunjOuTnOuKlCDrsKnsi50g4oCUIOy9lOuTnOuKlCDtm4Trs7Trp4jri6"
           "Qg66Gc6re47Jqw64+E7JeQIC0yIOulvCDqs7HtlbQg44CM64Ku7J2E7IiY66GdIOyi"
           "i+uLpOOAjSDroZwg67Cp7Zal7J2EIOuwmOyghOyLnO2CpOqzoCwg6rOE7IiYIOqwnO"
           "yImOyXkCBBSUMg64qUIDIg66W8LCBCSUMg64qUIGxvZyBuIOydhCDqs7HtlZwg7Yyo"
           "64SQ7YuwIO2VreydhCDrjZTtlanri4jri6QuIOygge2VqeydhCAz7ZqMIOqwmeydgC"
           "DsvZTrk5zsl5DshJwg7ZWY64uIIOq0gOy4oSDsp5HtlanCt+2DgOq5g8K37Jik7ZSE"
           "7IWL7J20IOyggOygiOuhnCDqsJnslYTsp4Dqs6AsIOq3uOuemOyVvCDshLgg7KCQ7I"
           "iY66W8IOqwmeydgCDquLDspIDsl5DshJwg67mE6rWQ7ZWgIOyImCDsnojsirXri4jr"
           "i6QuCg==",
           "과제":
           "8J+UkyDqs7zsoJwg7JmE7ISx67O4IOKAlCBBSSDsvZTrk5wg7Iuc7J6RIO2WiSDslY"
           "Trnpjsl5Ag65Ok7Ja07JmA7JW8IO2VoCDsvZTrk5zripQg7J20IOygleuPhOyeheuL"
           "iOuLpAoK4pagIOydtCDsvZTrk5zrpbwg67aA66W4IO2UhOuhrO2UhO2KuCDsmIgKCi"
           "AgICDsnbTrr7gg66eM65Ok7Ja07KC4IOyeiOuKlCBkZiwgQ0FORElEQVRFUywgRl9W"
           "RUgsIEZfVkVIX0JJTiwgb2ZmLCBTQU1QTEUzIOydhCDqt7jrjIDroZwg7JOw6rOgIO"
           "uLpOyLnCDrp4zrk6Tsp4DripQg66eQ7JWEIOykmC4gQ0FORElEQVRFUyDsl5Ag65Og"
           "CiAgICDtm4Trs7QgNOqwnChudWxsIMK3IHZlaGljbGUgwrcgY29yZSDCtyBmdWxsKe"
           "ydhCDrsJjrs7XrrLjsnLzroZwg7Y+s7JWE7IahIEdMTSDsl5Ag7KCB7ZWp7ZWY6rOg"
           "IOygge2VqSDqsrDqs7zrpbwg7J2066aE7Jy866GcIOq6vOuCtCDsk7gg7IiYIOyeiO"
           "qyjCDrqqjslYQg65GUIOuLpOydjCwg66qo7ZiVIOydtOumhCDCtwogICAg7KCI7Y64"
           "7J2EIO2PrO2VqO2VnCDqs4TsiJgg6rCc7IiYIMK3IOuhnOq3uOyasOuPhCDCtyDsnb"
           "Ttg4jrj4QgwrcgQUlDIMK3IEJJQyDCtyDqtIDsuKEg7IiY66W8IO2VnCDtlonslKkg"
           "64u07J2AIO2RnOulvCBBSUMg6rCAIOuCruydgCDsiJzsnLzroZwg7KCV66Cs7ZW0IO"
           "yduOyHhO2VtCDspJgg4oCUIOuFuOy2nCDrs7TsoJXsnYAgb2Zmc2V0CiAgICDsl5Ag"
           "b2ZmIOulvCDrhJjquLDrqbQg65CY6rOgIEFJQyDripQg66Gc6re47Jqw64+E7JeQIC"
           "0yIOulvCDqs7HtlZwg6rCS7JeQIOqzhOyImCDqsJzsiJjsnZggMuuwsOulvCDrjZTt"
           "lZwg6rCS7J207JW8LiDqt7gg64uk7J2MICLssKjrn4kg66qo7ZiV7J2YIEFJQyA9IC"
           "jsiKvsnpApIOygkCIsICLssKjrn4kg66qo7ZiV7J2YIOywqOuguQogICAgMeuFhCDs"
           "gqzqs6DsnKjruYQgPSAo7Iir7J6QKSDrsLAiLCAi6rKs67O4IOqzhOyVvSDshLgg6r"
           "G0IOyYiOy4oSDsmpTsnKjsnZgg7LWc64yAwrfstZzshowg67mEID0gKOyIq+yekCkg"
           "67CwIiDshLgg6rCS7J2EIOyGjOyImOygkCDslYTrnpggNOyekOumrOq5jOyngCDri6"
           "jsnITsmYAg7ZWo6ruYIOy2nOugpSDtlZwg7ZaJ7JSpIHByaW50IO2VtCDspJgg4oCU"
           "IOyCrOqzoOycqOu5hOuKlCDssKjrn4kg66qo7ZiVCiAgICDqs4TsiJjtkZzsl5DshJ"
           "wg7LCo66C5IO2VnCDtlonrp4wg6rOo6528IOyngOyImOulvCDslIzsmrQg6rCS7J20"
           "6rOgLCDtj63snYAgU0FNUExFMyDshLgg6rG07JeQIOywqOufiSDrqqjtmJXsnZgg7J"
           "iI7Lih7J2EIOqxuOyWtCDqsIDsnqUg64aS7J2AIOqwkuydhCDqsIDsnqUg64Ku7J2A"
           "IOqwkuycvOuhnCDrgpjriIgg67Cw7IiY7JW8LiDrhKQg7KCB7ZWp7J2YIOq0gOy4oQ"
           "ogICAg7IiY6rCAIOuqqOuRkCDqsJnsnYDsp4Ag7Lac66ClIO2VnCDtloksIEZfVkVI"
           "X0JJTiDsnLzroZwg7J207ZWtIEdMTSDsnYQg7ZWY64KYIOuNlCDrj4zroKQg7LCo66"
           "C5IDHrhYTsnZgg7IKs6rOg7Jyo67mE7JmAIOyYpOymiOu5hOulvCDrgpjrnoDtnogg"
           "64aT7J2AIOy2nOugpSDtlZwg7ZaJLCDssKjrn4kg66qo7ZiVIOqzhOyImOulvCDsoI"
           "TrtoAg7KeA7IiY66W8IOy3qO2VtCDrs7Xsm5DtlZwKICAgIOyCrOqzoOycqOu5hCDt"
           "kZwsIOq3uOumrOqzoCBTQU1QTEUzIOydmCDshLgg6rG07JeQIOuMgO2VtCDruIzrnp"
           "zrk5wgwrcg7LCo66C5IMK3IOuCmOydtCDCtyDtlaDspp3soJDsiJjsmYAg7LCo65+J"
           "IOuqqO2YlSDCtyDtlbXsi6wg66qo7ZiV7J2YIOyYiOy4oSDsl7DqsIQg7IKs6rOg7J"
           "yo7J2EIOuCmOuegO2eiCDrhpPsnYAg7ZGc64+EIOyduOyHhO2VtCDso7zqs6AuCiAg"
           "ICDqt7jrnpjtlITripQg65SxIO2VnCDsnqXrp4wg4oCUIOywqOufiSDrqqjtmJXqs7"
           "wg7ZW17IusIOuqqO2YleydtCBkZiDsoITssrTsl5Ag66ek6riw64qUIOuFuOy2nCAx"
           "64WEIOq4sOykgCDsmIjsuKEg7Jew6rCEIOyCrOqzoOycqOydhCDqsIHqsIEgMCDrto"
           "DthLAgMTAwIOq5jOyngOydmCDrsLHrtoTsnITroZwg64qY7Ja064aT7JWEIOqzoeyE"
           "oCDrkZjsnYQKICAgIOq3uOumrOqzoCwg7IS466Gc7LaV7J2AIOuhnOq3uCDriIjquI"
           "jsnYQg7JOw6rOgIOuRkCDstpXsl5Ag652867Ko7J2EIOuLrOqzoCDslrTripAg6rOh"
           "7ISg7J20IOyWtOuKkCDrqqjtmJXsnbjsp4Ag67KU66GA66GcIOq1rOu2hO2VtCDspJ"
           "guCgogICAgZml0cywgcm93cyA9IHt9LCBbXQogICAgZm9yIG5hbWUsIGZvcm11bGEg"
           "aW4gQ0FORElEQVRFUy5pdGVtcygpOiAgICAgICAgICAgICAgICAgIyDtm4Trs7QgNO"
           "qwnOulvCDtlZwg67CY67O166y47JeQ7IScIOygge2Vqe2VqeuLiOuLpAogICAgICAg"
           "IHIgPSBzbWYuZ2xtKGZvcm11bGE9Zm9ybXVsYSwgZGF0YT1kZiwgZmFtaWx5PXNtLm"
           "ZhbWlsaWVzLlBvaXNzb24oKSwgb2Zmc2V0PW9mZikuZml0KCkKICAgICAgICBrID0g"
           "aW50KHIuZGZfbW9kZWwpICsgMSAgICAgICAgICAgICAgICAgICAgICAgICMgZGZfbW"
           "9kZWwg7J2AIOygiO2OuOydhCDruoAg6rCc7IiY6528IDEg7J2EIOuNlO2VqeuLiOuL"
           "pAogICAgICAgIG4gPSBpbnQoci5ub2JzKQogICAgICAgIGZpdHNbbmFtZV0gPSByCi"
           "AgICAgICAgcm93cy5hcHBlbmQoeyJtb2RlbCI6IG5hbWUsICJrIjogaywgImxvZ2xp"
           "ayI6IHIubGxmLCAiZGV2aWFuY2UiOiByLmRldmlhbmNlLAogICAgICAgICAgICAgIC"
           "AgICAgICAiYWljIjogLTIgKiByLmxsZiArIDIgKiBrLAogICAgICAgICAgICAgICAg"
           "ICAgICAiYmljIjogLTIgKiByLmxsZiArIG5wLmxvZyhuKSAqIGssICJub2JzIjogbn"
           "0pCgogICAgc2NvcmVfdGFibGUgPSBwZC5EYXRhRnJhbWUocm93cykuc29ydF92YWx1"
           "ZXMoImFpYyIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHByaW50KHNjb3JlX3"
           "RhYmxlLnJvdW5kKDIpLnRvX3N0cmluZyhpbmRleD1GYWxzZSkpCgogICAgdmVoX2Zp"
           "dCA9IGZpdHNbInZlaGljbGUiXQogICAgYWljX2J5X21vZGVsID0gZGljdCh6aXAoc2"
           "NvcmVfdGFibGVbIm1vZGVsIl0sIHNjb3JlX3RhYmxlWyJhaWMiXSkpCiAgICBwcmlu"
           "dChmIuywqOufiSDrqqjtmJXsnZggQUlDIDoge2FpY19ieV9tb2RlbFsndmVoaWNsZS"
           "ddOiwuMmZ9IOygkCIpCiAgICBwcmludChmIuywqOufiSDrqqjtmJXsnZgg7LCo66C5"
           "IDHrhYQg7IKs6rOg7Jyo67mEIDoge2Zsb2F0KG5wLmV4cCh2ZWhfZml0LnBhcmFtc1"
           "snVmVoQWdlJ10pKTouNGZ9IOuwsCIpCgogICAgcHJlZDMgPSB2ZWhfZml0LnByZWRp"
           "Y3QoU0FNUExFMykgICAgICAgICAgICAgICAgICAgICAgICAgICMg64W47Lac7J2EIO"
           "yViCDrhJjquLDrqbQg44CM64W47LacIDHrhYQg6riw7KSAIOyalOycqOOAjeuhnCDr"
           "gpjsmLXri4jri6QKICAgIHByaW50KGYi6rKs67O4IOqzhOyVvSDshLgg6rG0IOyYiO"
           "y4oSDsmpTsnKjsnZgg7LWc64yAwrfstZzshowg67mEIDoge2Zsb2F0KHByZWQzLm1h"
           "eCgpIC8gcHJlZDMubWluKCkpOi40Zn0g67CwIikKICAgIHByaW50KCLrhKQg7KCB7Z"
           "Wp7J2YIOq0gOy4oSDsiJjqsIAg66qo65GQIOqwmeydgOqwgCA6Iiwgc2NvcmVfdGFi"
           "bGVbIm5vYnMiXS5udW5pcXVlKCkgPT0gMSkKCiAgICBiaW5vbV9maXQgPSBzbWYuZ2"
           "xtKGZvcm11bGE9Rl9WRUhfQklOLCBkYXRhPWRmLCBmYW1pbHk9c20uZmFtaWxpZXMu"
           "Qmlub21pYWwoKSkuZml0KCkgICMg7Jqw67OA7J2AIOq3uOuMgOuhnCwg7YOA6rmD66"
           "eMIOyCrOqzoCDsl6zrtoAKICAgIHByaW50KGYi7LCo66C5IDHrhYQg4oCUIOyCrOqz"
           "oOycqOu5hCB7bnAuZXhwKHZlaF9maXQucGFyYW1zWydWZWhBZ2UnXSk6LjRmfSDrsL"
           "AiCiAgICAgICAgICBmIiDCtyDsmKTspojruYQge25wLmV4cChiaW5vbV9maXQucGFy"
           "YW1zWydWZWhBZ2UnXSk6LjRmfSDrsLAiKQoKICAgIHJhdGVfbXVsdCA9IG5wLmV4cC"
           "h2ZWhfZml0LnBhcmFtcy5kcm9wKCJJbnRlcmNlcHQiKSkgICAgICAgICAgICAgICMg"
           "7LCo65+JIOuqqO2YlSDqs4TsiJggLT4g7IKs6rOg7Jyo67mECiAgICBwcmludChyYX"
           "RlX211bHQucm91bmQoNCkudG9fc3RyaW5nKCkpCgogICAgc2FtcGxlX3RhYmxlID0g"
           "cGQuRGF0YUZyYW1lKHsi67iM656c65OcIjogU0FNUExFM1siVmVoQnJhbmQiXS5hc3"
           "R5cGUoc3RyKS50b19udW1weSgpLCAgICMg44CM7KSA67mEIOyFgOOAjeydtCDrvZHs"
           "lYQg65GUIOyEuCDqsbQKICAgICAgICAgICAgICAgICAgICAgICAi7LCo66C5IjogU0"
           "FNUExFM1siVmVoQWdlIl0udG9fbnVtcHkoKSwKICAgICAgICAgICAgICAgICAgICAg"
           "ICAi64KY7J20IjogU0FNUExFM1siRHJpdkFnZSJdLnRvX251bXB5KCksCiAgICAgIC"
           "AgICAgICAgICAgICAgICAgIu2VoOymneygkOyImCI6IFNBTVBMRTNbIkJvbnVzTWFs"
           "dXMiXS50b19udW1weSgpLAogICAgICAgICAgICAgICAgICAgICAgICLssKjrn4kg66"
           "qo7ZiVIjogbnAuYXNhcnJheShwcmVkMywgZHR5cGU9ZmxvYXQpLAogICAgICAgICAg"
           "ICAgICAgICAgICAgICLtlbXsi6wg66qo7ZiVIjogbnAuYXNhcnJheShmaXRzWyJjb3"
           "JlIl0ucHJlZGljdChTQU1QTEUzKSwgZHR5cGU9ZmxvYXQpfSkKICAgIHByaW50KHNh"
           "bXBsZV90YWJsZS5yb3VuZCg0KS50b19zdHJpbmcoaW5kZXg9RmFsc2UpKSAgICAgIC"
           "AgICAgICAgICAjIOyYiOy4oSDsmpTsnKjsnYAg64W47LacIDHrhYQg6riw7KSALCDq"
           "sbQv64WECgogICAgcmF0ZV9ieV9tb2RlbCA9IHttOiBucC5hc2FycmF5KGZpdHNbbV"
           "0ucHJlZGljdChkZikpIGZvciBtIGluICgidmVoaWNsZSIsICJjb3JlIil9ICAgIyDs"
           "mKTtlITshYsg7JeG7J20ID0g64W47LacIDHrhYQg6riw7KSACiAgICBwID0gbnAuYX"
           "JhbmdlKDAsIDEwMSkKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0o"
           "RklHX1csIDQpKQogICAgYXgucGxvdChwLCBucC5wZXJjZW50aWxlKHJhdGVfYnlfbW"
           "9kZWxbInZlaGljbGUiXSwgcCksIGNvbG9yPUNPTFsi7ZWY64qY7IOJIl0sIGx3PTIs"
           "IGxhYmVsPSLssKjrn4kg66qo7ZiVIikKICAgIGF4LnBsb3QocCwgbnAucGVyY2VudG"
           "lsZShyYXRlX2J5X21vZGVsWyJjb3JlIl0sIHApLCBjb2xvcj1DT0xbIuuCqOyDiSJd"
           "LCBsdz0yLCBsYWJlbD0i7ZW17IusIOuqqO2YlSIpCiAgICBheC5zZXRfeXNjYWxlKC"
           "Jsb2ciKQogICAgYXguc2V0X3hsYWJlbCgi6rOE7JW97J2EIOyYiOy4oSDsmpTsnKgg"
           "7Iic7Jy866GcIOuKmOyWtOuGk+ydgCDrsLHrtoTsnIQgKCUpIikKICAgIGF4LnNldF"
           "95bGFiZWwoIuyYiOy4oSDsl7DqsIQg7IKs6rOg7JyoICjqsbQv64WELCDrhbjstpwg"
           "MeuFhCDquLDspIAsIOuhnOq3uCDriIjquIgpIikKICAgIGF4LmxlZ2VuZCgpCiAgIC"
           "BwbHQuc2hvdygpCgrtlbTshJ0KCi0g7LCo65+JIOuqqO2YleydmCBBSUMg64qUIDI5"
           "MSwyODIuMzgg7KCQ7J6F64uI64ukLiDquLDspIDshKAobnVsbCkgMjkzLDI1MC43Ni"
           "Drs7Tri6QgMSw5NjguMzcg7KCQIOuCruycvOuLiCDssKjrn4kg7KCV67O06rCAIOyC"
           "rOqzoOycqOydhCDshKTrqoXtlZjquLDripQg7ZWp64uI64ukIOKAlCDstpzroKXCt+"
           "ywqOugucK37Jew66OMwrfruIzrnpzrk5zrp4zsnLzroZzrj4Qg6rOE7JW97J2EIOyc"
           "hO2XmCDsiJzsnLzroZwg6rWs67aE7ZWY64qUIO2emOydtCDsnojri6TripQg65y77J"
           "6F64uI64ukLgotIOuLpOunjCDtlbXsi6wg66qo7ZiVIDI4OCwxNDkuODEg67O064uk"
           "IEFJQyDqsIAgMywxMzIuNTfsoJAg64aS7Iq164uI64ukKO2ZleyepSDrqqjtmJUgMj"
           "g3LDI3Mi44MyDqs7zripQgNCwwMDkuNTXsoJAg7LCo7J20KS4g7ZuE67O0IDTqsJzr"
           "pbwg6rCZ7J2AIOq4sOykgOyXkOyEnCDruYTqtZDtlZjrqbQg7LCo65+JIOuqqO2Yle"
           "ydtCDquLDspIDshKDqs7wg7ZW17IusIOuqqO2YlSDsgqzsnbTsl5Ag7JyE7LmY7ZWp"
           "64uI64ukIOKAlCDrgpjsnbTsmYAg7ZWg7Kad7KCQ7IiY6rCAIOuLtOqzoCDsnojrjZ"
           "gg7KCV67O066W8IOywqOufiSDsoJXrs7TroZzripQg64yA7Iug7ZWY7KeAIOuqu+2V"
           "nOuLpOuKlCDrnLvsnoXri4jri6QuIOyyqyDtmZTrqbQg6rKs7KCB7J2YIO2VnOqzhO"
           "qwgCDsl6zquLDshJwg64KY7Ji164uI64ukLgotIOqzhOyImCDqsJzsiJjripQgMTTq"
           "sJzsnoXri4jri6Qg4oCUIOygiO2OuCAxICsgVmVoUG93ZXIgMSArIFZlaEFnZSAxIC"
           "sgVmVoR2FzIDEgKyBWZWhCcmFuZCAxMC4g67iM656c65OcIDEx6rCcIOyImOykgOyX"
           "kOyEnCDquLDspIAg7ZWY64KY66W8IOu6gCAxMOqwnOqwgCDtlZzqurzrsojsl5Ag65"
           "Ok7Ja07Jik64qUIOqyg+ydtCDsnbQg66qo7ZiV7J20IOustOqxsOybjOyngOuKlCDs"
           "nbTsnKDqs6AsIEFJQyDripQg6re4IDEw6rCc7JeQIDIw7KCQLCBCSUMg64qUIDEzNC"
           "4yN+ygkOydhCDrtoDqs7ztlanri4jri6QuIOq3uOuemOyEnCBCSUMg66GcIOyInOyc"
           "hOulvCDrp6Tqsqjrj4Qg7LCo65+JIOuqqO2YlSAyOTEsNDQyLjM2IOydgCDtlbXsi6"
           "wg66qo7ZiVIDI4OCwyNTIuNjUg65Kk7JeQIOq3uOuMgOuhnCDrgqjsirXri4jri6Qu"
           "Ci0g7LCo66C5IOuwsOycqOydtCAxIOuztOuLpCDsnpHsirXri4jri6Qg4oCUIOywqO"
           "ugueydtCAx64WEIOymneqwgO2VoOyImOuhnSDsgqzqs6DsnKjsnbQg7KGw6riI7JSp"
           "IOqwkOyGjO2VnOuLpOuKlCDrnLvsnbTqs6AsIOyalOycqO2RnOuhnOuKlCDjgIzssK"
           "jroLkgMeuFhOuLuSDslb0gMy40JSDtlaDsnbjjgI0g7J6F64uI64ukLiDri6Trp4wg"
           "7J20IOqwkuydgCDruIzrnpzrk5zCt+y2nOugpcK37Jew66OM66W8IOqzoOygle2VnC"
           "DssYQg7J2964qUIOuztOygleuQnCDtmqjqs7zsnoXri4jri6QuIOyasOuzgOyXkOyE"
           "nCDruIzrnpzrk5zrpbwg7KCc7Jm47ZWY6rOgIOuLpOyLnCDsoIHtlantlZjrqbQg6r"
           "CZ7J2AIOuNsOydtO2EsOyXkOyEnOuPhCDri6Trpbgg6rCS7J20IOuCmOyYteuLiOuL"
           "pCDigJQg44CM7J2066GgIDYuIOyLpOygnCDrjbDsnbTthLAg7KCB7Jqp44CN7J2YIO"
           "uCmOydtCDrsLDsnKjsnbQgMTDrhYTri7kgMC45NTQg7JmAIDEuMDc0IOuhnCDri6zr"
           "nbzsoYzrjZgg7IKs66GA6rCAIOq3uCDspp3qsbDsnoXri4jri6QuIOq3uOuemOyEnC"
           "Dtmozsi6Dsl5Ag67Cw7Jyo7J2EIOyggeydhCDrlYzripQg66y07JeH6rO8IO2VqOq7"
           "mCDsoIHtlantlZwg6rOE7IiY7J247KeA66W8IO2VqOq7mCDsoIHqs6AsIOydtCDrsL"
           "DsnKjsl5Ag7Iug66Kw6rWs6rCE7J2EIOyWvOuniOuCmCDrhJPqsowg67aZ7JesIOuz"
           "tOqzoO2VtOyVvCDtlZjripTsp4DripQg64K07J28IOq1rO2VqeuLiOuLpC4KLSDqs4"
           "TsiJjrpbwg7JqU7JyoIOusuOyepeycvOuhnCDsmK7quLDrqbQg7J2066CH7Iq164uI"
           "64ukLiDssKjroLnsnYAgMeuFhOuLuSAwLjk2NTjrsLDroZwg7ZW066eI64ukIDMuNC"
           "Ug7JSpIOqwkOyGjO2VmOqzoCwg7ZyY67Cc7Jyg7LCo6rCAIOqyveycoOywqCDrjIDr"
           "uYQgMS4xMDE567Cw7J6F64uI64ukLiDruIzrnpzrk5zripQg6riw7KSAIEIxIOuMgO"
           "u5hCBCMTIg6rCAIDEuMjI5N+uwsOuhnCDqsIDsnqUg64aS6rOgIEIxNCDqsIAgMC44"
           "MzQ567Cw66GcIOqwgOyepSDrgq7sirXri4jri6QuIOy2nOugpShWZWhQb3dlcinsnY"
           "AgMS4wMDEz67Cw66GcIOyCrOyLpOyDgSDrs4DtmZTqsIAg7JeG7Iq164uI64ukLiDq"
           "sJnsnYAg7Jqw67OA7Jy866GcIOyCrOqzoCDsl6zrtoAoQ2xhaW1OYiA+IDAp66W8IO"
           "unnuy2mCDsnbTtla0gR0xNIOyXkOyEnOuKlCDssKjroLkgMeuFhOydmCDsmKTspojr"
           "uYTqsIAgMC45NzM0IOuhnCDrgpjsmKTripTrjbAsIOywqOuguSDtlZwg7ZaJ7J2YIO"
           "yCrOqzoOycqOu5hCAwLjk2NTgg6rO8IOqwkuuPhCDrnLvrj4Qg64uk66aF64uI64uk"
           "IOKAlCDtlZjrgpjripQg7IKs6rOg66W8IOuCvCDsmKTspojsl5AsIOuLpOuluCDtlZ"
           "jrgpjripQg7Jew6rCEIOyCrOqzoOycqOyXkCDqs7HtlbTsp4DripQg67Cw7Jyo7J6F"
           "64uI64ukLiDsgqzqs6Drpbwg64K4IOqzhOyVveydtCA1LjAlIOu/kOydtOudvCDrkZ"
           "Ag67Cw7Jyo7J20IOqwgOq5jOybjCDrs7Tsnbwg67+Q7J6F64uI64ukLgotIOq3uOum"
           "vOydtCDssYTrhJDtjIAg7ZqM7Iug7J2YIOOAjO2VnOqzhOOAjSDtlZwg66y47J6l7J"
           "2EIOq3uOuMgOuhnCDrp4zrk6TslrQg7KSN64uI64ukLiDquLDspIDsnYAg64W47Lac"
           "IDHrhYQo7Jik7ZSE7IWLIDAp7Jy866GcIOyYiOy4oe2VnCDsl7DqsIQg7IKs6rOg7J"
           "yo7J6F64uI64ukIOKAlCDqs4Tslb0g7ZaJ7J2EIOuEo+yWtCBwcmVkaWN0IOulvCDr"
           "toDrpbTrqbQg7Jik7ZSE7IWL7J20IDAg7Jy866GcIOuGk+yeheuLiOuLpC4g7KCB7Z"
           "WpIOuVjCDrp4zrk6TslrTsp4QgZml0dGVkdmFsdWVzIOulvCDqt7jrjIDroZwg7JOw"
           "66m0IOq3uCDqs4Tslb3snZgg64W47Lac7J20IOqzse2VtOynhCDqsbTsiJjrnbwg7J"
           "qU7Jyo7J20IOyVhOuLiOqzoCwg64W47LacIDAuM+uFhCDqs4Tslb3snbTrqbQg7JqU"
           "7Jyo7J2YIDMwJSDroZwg7Lac66Cl65Cp64uI64ukLiA1fjk1IOuwseu2hOychOuKlC"
           "DssKjrn4kg66qo7ZiV7J20IDAuMDY4NH4wLjE1MTUg6rG0L+uFhCwg7ZW17IusIOuq"
           "qO2YleydtCAwLjA2ODV+MC4yMDEzIOqxtC/rhYTsnLzroZwg6rCA7Jq0642w6rCAIO"
           "qxsOydmCDqsrnsuZjripTrjbAsIOqzoeyEoOydmCDsmKTrpbjsqr0g64Gd7J2AIDAu"
           "MTU5MCDrjIAgNS45OTYxIOuhnCDtgazqsowg7LCo7J206rCAIOuCqeuLiOuLpC4g7J"
           "m87Kq9IOunqCDrgZ3rj4QgMC4wMDM1IOuMgCAwLjA1NTgg66GcIOuLrOudvOyngOuK"
           "lOuNsCwg7J2064qUIOywqOugueydtCDslYTso7wg66eO7J2AIOqzhOyVvSjstZzqs6"
           "AgMTAw64WEKeyXkCDssKjrn4kg66qo7ZiV7J20IOyngOuCmOy5mOqyjCDrgq7snYAg"
           "6rCS7J2EIOyCsOy2nO2VmOq4sCDrlYzrrLjsnoXri4jri6QuIOydveydhCDqsoPsnY"
           "Ag7Jik66W47Kq9IOuBneyeheuLiOuLpCDigJQg7LCo65+JIOygleuztOunjOycvOuh"
           "nOuKlCDqs6DsnITtl5gg6rOE7JW97J2EIOqzoOychO2XmOycvOuhnCDsgrDstpztlZ"
           "jsp4Ag66q77ZWc64uk64qUIOucu+ydtOqzoCwg7LKrIO2ZlOuptCDqsqzsoIHsnbQg"
           "6rCA7J6FIOuSpOydmCDsmpTsnKjrs7Tri6Qg64Ku6rKMIOuCmOyYpOuKlCDsqr3snL"
           "zroZwg7Ja06riL64Kp64uI64ukLgotIOqyrOuzuCDqs4Tslb0g7IS4IOqxtOydhCDr"
           "gpjrnoDtnogg64aT7Jy866m0IOq3uCDssKjsnbTqsIAg6riI7JWhIOywqOydtOuhnC"
           "Drgpjtg4Drgqnri4jri6QuIOywqOufiSDrqqjtmJXsnbQg66ek6riw64qUIOyalOyc"
           "qOydgCDjgIzqsIDjgI0gMC4xNTgwIMK3IOOAjOuCmOOAjSAwLjA5NzUgwrcg44CM64"
           "uk44CNIDAuMDc0OCDqsbQv64WE7J206528IO2PreydtCAyLjEx67Cw7J24642wLCDq"
           "sJnsnYAg7IS4IOqxtOydhCDtlbXsi6wg66qo7ZiV7JeQIOyggeyaqe2VmOuptCAwLj"
           "IwNDYgwrcgMC4wOTA5IMK3IDAuMDcyOCDroZwgMi44MeuwsOq5jOyngCDsu6Tsp5Hr"
           "i4jri6QuIOqzhOyVvSDsoITssrTroZwg6rOE7IKw7ZWY66m0IDbrsogg6rOh7ISg7J"
           "2YIDk167aE7JyEIMO3IDXrtoTsnITqsIAg7LCo65+JIOuqqO2YlSAyLjIx67CwLCDt"
           "lbXsi6wg66qo7ZiVIDIuOTTrsLDsnoXri4jri6QuIO2PreydtCDsooHri6TripQg6r"
           "KD7J2AIOychO2XmO2VnCDqs4Tslb3sl5Drj4Qg64aS7J2AIOqwkuydhCDsgrDstpzt"
           "lZjsp4Ag66q77ZWc64uk64qUIOucu+ydtOudvCjsnbTroaAgNC4g7Y+s7JWE7IahIO"
           "2ajOq3gCksIOyyqyDtmZTrqbQg6rKs7KCB7J2AIOOAjOyInOychOuKlCDrjIDstqkg"
           "66ee7KeA66eMIOq4iOyVoeydmCDtj63snbQg7KKB7JWE7KeEIOqwkuOAjeydtCDrkK"
           "nri4jri6QuIOqyrOuzuCDjgIzqsIDjgI0g64qUIDMw7IS4wrftlaDspp0gMTAw7KCQ"
           "7J206528IOyCrOuejCDsoJXrs7TqsIAg642U7ZW07KeA64qUIOyInOqwhCDsmpTsnK"
           "jsnbQg7IOB7Iq57ZWY64qU642wLCDssqsg7ZmU66m07J2AIOq3uOqyg+ydhCDrqrsg"
           "66y87Ja067SQ7IScIDEuM+uwsCDrgq7qsowg7IKw7Lac7ZWp64uI64ukLgoK7L2U65"
           "Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudIOKAlCDsvZTrk5zripQg7ZuE67O0"
           "IDTqsJzrpbwg7ZWcIOuwmOuzteusuOyXkOyEnCDqsJnsnYAg67aE7Y+swrfqsJnsnY"
           "Ag7Jik7ZSE7IWL7Jy866GcIOygge2Vqe2VtCDqtIDsuKEg7KeR7ZWpwrftg4DquYPC"
           "t+yYpO2UhOyFi+ydhCDsoIDsoIjroZwg66ee7LaYIOuLpOydjCwg66Gc6re47Jqw64"
           "+E7JeQIC0yIOulvCDqs7HtlbQg44CM64Ku7J2E7IiY66GdIOyii+uLpOOAjSDroZwg"
           "67Cp7Zal7J2EIOuwmOyghOyLnO2CpOqzoCDqs4TsiJgg6rCc7IiY66eM7YG8IO2MqO"
           "uEkO2LsCDtla3snYQg642U7ZWp64uI64ukLiDqt7jrnpjshJwg64SkIOygkOyImOul"
           "vCDqsJnsnYAg6riw7KSA7JeQ7IScIOu5hOq1kO2VoCDsiJgg7J6I6rKMIOuQmOqzoC"
           "wg6rCZ7J2AIOygge2VqeyXkOyEnCDsmKTtlITshYsg7JeG7J20IOyYiOy4oeydhCDr"
           "vZHsnLzrqbQg6re4IOuqqO2YleydtCDssqsg7ZmU66m07JeQIOunpOq4uCDsiJgg7J"
           "6I64qUIOqwkuydmCDtj63snbQg6re464yA66GcIOqzoeyEoOydtCDrkKnri4jri6Qu"
           "Cg=="
}
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
"eyJtMSI6IHsi7Jew6rCEIOyCrOqzoOycqCAo6rG0L+uFhCkiOiBbIuuhnOq3uCDrp4"
           "Htgawg66qo7ZiV7J2YIOqzhOyImOuKlCDroZzqt7gg7Iqk7LyA7J287JeQIOyeiOyK"
           "teuLiOuLpCDigJQg7KCI7Y647J2EIOq3uOuMgOuhnCDsnb3snLzrqbQg7J2M7IiY6r"
           "CAIOuCmOyYpOqzoCwg7KeA7IiY66GcIOuzgO2ZmO2VtOyVvCDqsbQv64WEIOydmCDs"
           "mpTsnKjsnbQg65Cp64uI64ukLiDqsJLsnbQg7KCI67CY7K+kKOqzhOyVveuLuSDsgq"
           "zqs6Ag6rG07IiYKeycvOuhnCDrgq7slYTsoYzri6TrqbQg7Jik7ZSE7IWL7J20IOu5"
           "oOynhCDqsoPsnoXri4jri6QiLCAi7J20IOy9lOuTnOqwgCDsoIjtjrjsnYQg66y07J"
           "eH7Jy866GcIOuzteybkO2WiOqzoCBvZmZzZXQg7JeQIOustOyXh+ydhCDrhJjqsrzr"
           "ipTqsIAiXSwgIm51bGwg66qo7ZiV7J2YIOuhnOq3uOyasOuPhCI6IFsi7KCB7ZWpIO"
           "qysOqzvCDqsJ3ssrTqsIAg65Ok6rOgIOyeiOuKlCDroZzqt7jsmrDrj4Trpbwg64uo"
           "7JyEIOyXhuydtCDstpzroKUg7ZWcIO2WiSDrjZQg7J247IeE7ZWY66m0IOuQqeuLiO"
           "uLpCDigJQg7Ja07KCcIOy1nOyGjO2ZlO2VnCDsnYzsnZgg66Gc6re47Jqw64+E7JmA"
           "IOu2gO2YuOunjCDri6Trpbgg6rCS7J206528IOydjOyImOuhnCDstpzroKXrkKnri4"
           "jri6QiLCAi7J20IOuqqO2YleydtCDstZzrjIDtmZTtlZwg6rKD7J20IOustOyXh+yd"
           "uOqwgCJdLCAicGxvdCI6IFsi6rCA66Gc7LaV7J20IO2VoOymneygkOyImCDqtazqsI"
           "Qg7KSR7JWZ6rCS7J2EIOyymOydjOu2gO2EsCDrgZ3quYzsp4Ag7Y+s7ZWo7ZWY64qU"
           "7KeAIMK3IOq6vuydgOyEoOydtCDtlZjrgpjsnbjsp4Agwrcg7IS466Gc7LaV7J20IO"
           "uhnOq3uCDriIjquIjsnbjsp4Agwrcg66qo7ZiV7J20IOunpOq4sOuKlCDri6jsnbwg"
           "7JqU7Jyo7J2EIOqwgOuhnOyEoOycvOuhnCDstpTqsIDtlojripTsp4Agwrcg65GQIO"
           "y2leyXkCDrnbzrsqjsnYQg64us7JWY64qU7KeAIOywqOuhgOuhnCDrs7TshLjsmpQi"
           "LCAi64K0IOq3uOumvOydmCDslrTripAg7LaVwrfslrTripAg7ISg7J20IOq4sOykgO"
           "qzvCDrp57sp4Ag7JWK64qU7KeAIl0sICJlcnJvciI6IFsi7Jik66WYIOusuOyepSDs"
           "sqsg7ZaJ7J2EIOq3uOuMgOuhnCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg44CM7K"
           "SA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aEKGRmIMK3IG9mZiDCtyBibV94IMK3"
           "IGJtX3JhdGUp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsk7"
           "Drnbzqs6Ag67aE66qF7Z6IIOyggeycvOyEuOyalCIsICLsnbQg7Jik66WY6rCAIOyW"
           "tOuKkCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwgIm0yIjogeyLtlaDspp3soJ"
           "DsiJggMeygkOydmCDsgqzqs6DsnKjruYQgKOuwsCkiOiBbWyIx7LCoIOKAlCBBSSDs"
           "vZTrk5wg7Iuc7J6RIO2WiSDslYTrnpjsl5DshJwg6rOE7IiY66W8IOuwsOycqOuhnC"
           "DrsJTqvrjripQg7L2U65OcIO2WiSDtlZjrgpjrp4wg7LC+7JWEIOuztOyEuOyalCAo"
           "cGN0IOyZgCByciDrpbwg66eM65Oc64qUIDLqsJwg7ZaJIOyCrOydtOyeheuLiOuLpC"
           "kiLCAiMuywqCDigJQg6re4IO2WieyXkOuKlCDsp4DsiJjrpbwg7Leo7ZWY64qUIOyX"
           "sOyCsOydtCDruaDsoLgg7J6I7Iq164uI64ukLiDjgIwxICsgzrLjgI0g64qUIM6yIO"
           "qwgCDsnpHsnYQg65WM66eMIO2Gte2VmOuKlCDqt7zsgqzrnbwg7YGwIOqzhOyImOyX"
           "kOyEnCDssKjsnbTqsIAg7Luk7KeR64uI64ukIl0sICLsnbQg7L2U65Oc7J2YIOyWtO"
           "uKkCDtlonsnbQg6rOE7IiY66W8IOuwsOycqOuhnCDsl63rs4DtmZjtlZjqs6Ag7J6I"
           "64qU6rCAIl0sICLrj4Tsi5ztmZQg65Ox6riJIEYg7J2YIOyCrOqzoOycqOu5hCAo67"
           "CwKSI6IFtbIjHssKgg4oCUIOqzhOyImOqwgCDtgbAg7ZWt66qp6rO8IOyekeydgCDt"
           "la3rqqnsl5Ag6rCZ7J2AIOq3nOy5meydtCDqsbjroLjripTsp4Ag67O07IS47JqULi"
           "DtlaDspp3soJDsiJjripQg7Ya16rO87ZaI64qU642wIOydtCDtla3rqqnrp4wg66ee"
           "7KeAIOyViuyVmOuLpOuptCDqt5zsuZnsnbQg6rOE7IiYIO2BrOq4sOyXkCDrlLDrnb"
           "wg64us65287KeA6rOgIOyeiOuKlCDqsoPsnoXri4jri6QiLCAiMuywqCDigJQg66Gc"
           "6re4IOyKpOy8gOydvOydmCDqs4TsiJjrpbwg7JuQ656YIOuLqOychOuhnCDrs7Xsm5"
           "DtlZjripQg6rec7LmZ7J2AIOqzhOyImOydmCDtgazquLDsmYAg7IOB6rSA7JeG7J20"
           "IOyngOyImCDtlZjrgpjsnoXri4jri6QuIOqzhOyImOqwgCDsnpHsnYQg65WM66eMIO"
           "2Gte2VmOuKlCDqt7zsgqzrpbwg7JOw66m0IO2BsCDqs4TsiJjsl5DshJwg7J2066CH"
           "6rKMIOywqOydtOqwgCDsu6Tsp5Hri4jri6QiXSwgIuqzhOyImOqwgCDtgbAg7ZWt66"
           "qp6rO8IOyekeydgCDtla3rqqnsl5Ag6rCZ7J2AIOuzgO2ZmCDqt5zsuZnsnbQg6rG4"
           "66C464qU6rCAIl0sICJwbG90IjogW1siMeywqCDigJQg6re466a87J2AIEFJIOy9lO"
           "uTnCDsi5zsnpEg7ZaJIOyVhOuemCDsvZTrk5zqsIAg7JWE7KeBIOunjOuTpOyngCDs"
           "lYrsirXri4jri6QuIOqwmeydgCDtlITroaztlITtirjsl5DshJwg7ZWo6ruYIOyLnO"
           "y8sOuKlOyngOu2gO2EsCDrs7TshLjsmpQiLCAiMuywqCDigJQg7KCI7Y647J2EIOu6"
           "gCA46rCc66eMIOyEuOuhnOy2leyXkCDsnojripTsp4Agwrcg66eJ64yAIOq4uOydtO"
           "qwgCDrsLDsnKjsnbjsp4Agwrcg67Cw7JyoIDEuMCDsl5Ag7IS466Gc7ISg7J2EIOq3"
           "uOyXiOuKlOyngCDCtyDqsIDroZzstpXsl5Ag652867Ko7J2EIOuLrOyVmOuKlOyngC"
           "DssKjroYDroZwg67O07IS47JqUIl0sICLrgrQg6re466a87J2YIOunieuMgCDqsJzs"
           "iJjsmYAg7IS466Gc7ISg7J20IOq4sOykgOqzvCDrp57sp4Ag7JWK64qU7KeAIl0sIC"
           "JlcnJvciI6IFsi7Jik66WYIOusuOyepSDssqsg7ZaJ7J2EIOq3uOuMgOuhnCDtipzt"
           "hLDsl5Ag67O07JesIOyjvOqzoCwg44CM7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J"
           "2066aEKGRmIMK3IEZfQ09SRSDCtyBvZmYgwrcgQ0xBSU1TX0FMTCnsnYAg64uk7Iuc"
           "IOunjOuTpOyngCDrp5Dqs6Ag6re464yA66GcIOyTsOudvOqzoCDrtoTrqoXtnogg7K"
           "CB7Jy87IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkOyEnCDsmZwg"
           "64Ks64qU7KeAIl19LCAibTMiOiB7IuqwgOyepSDrgq7snYAgQUlDICjsoJApIjogWy"
           "JBSUMg66W8IOustOyXh+ycvOuhnCDrp4zrk6Tsp4AgUFJPTVBUIOyXkCDsoIHsl4jr"
           "ipTsp4Ag67O07IS47JqUIOKAlCDroZzqt7jsmrDrj4TsmYAg7KCI7Y646rmM7KeAIO"
           "2PrO2VqO2VnCDqs4TsiJgg6rCc7IiY66GcIOqzhOyCsO2VnOuLpOuKlCDqsoMsIDPq"
           "sJwg7KSRIOqwgOyepSDsnpHsnYAg6rCS7J2EIOyWtOuKkCDrqqjtmJXsnbjsp4DsmY"
           "Ag7ZWo6ruYIO2VnCDtlonsnLzroZwg7Lac66Cl7ZWc64uk64qUIOqygywg64uo7JyE"
           "IOygkOyXkCDshozsiJjsoJAg7JWE656YIDLsnpDrpqzquYzsp4AuIiwgIuuCtCDsoJ"
           "DsiJjqsIAg66Gc6re47Jqw64+E7JmAIOqzhOyImCDqsJzsiJgg7KSRIOyWtOuKkCDs"
           "qr3sl5DshJwg7Ja06riL64Ks64qU6rCAIl0sICLssYTsoJDtkZwgbnVsbCDtlonsnZ"
           "gg66Gc6re47Jqw64+EIjogWyLtm4Trs7QgM+qwnOulvCDtlZwg67CY67O166y47JeQ"
           "7IScIOqwmeydgCDrtoTtj6zCt+qwmeydgCDtg4DquYPCt+qwmeydgCDsmKTtlITshY"
           "vsnLzroZwg7KCB7ZWp7ZWY65286rOgIOyggeyXiOuKlOyngCDrs7TshLjsmpQuIG51"
           "bGwg7ZaJIOqwkuydgCDrr7jshZggMSDsl5DshJwg7Lac66Cl7ZWcIOq4sOykgOyEoC"
           "DqsJLqs7wg7J287LmY7ZW07JW8IO2VqeuLiOuLpC4iLCAi7IS4IOygge2VqeyXkCDs"
           "mKTtlITshYvsnbQg64ukIOuTpOyWtOqwlOqzoCDtg4DquYPsnbQg66qo65GQIOqwme"
           "ydgOqwgCJdLCAicGxvdCI6IFsi6re4656Y7ZSEIOyalOq1rOulvCBQUk9NUFQg7JeQ"
           "IO2VmOuCmOyUqSDsoIHsl4jripTsp4Ag67O07IS47JqUIOKAlCDrp4nrjIAgM+qwnC"
           "wg6rCA66Gc7LaVIOuIiOq4iCDsnbTrpoTsnYAgYG51bGxgIMK3IGBjb3JlYCDCtyBg"
           "ZnVsbGAg6re464yA66GcLCDqsIDsnqUg64Ku7J2AIOunieuMgCDtlZjrgpjrp4wg7I"
           "OJ7J2EIOuLrOumrCwg6rCA66Gc7LaVwrfshLjroZzstpUg7J2066aE6rmM7KeALiIs"
           "ICLrgrQg6re466a87J2YIOunieuMgCDqsJzsiJjsmYAg64iI6riIIOydtOumhOydtC"
           "DquLDspIDqs7wg66ee7KeAIOyViuuKlOyngCJdLCAiZXJyb3IiOiBbIuyYpOulmCDr"
           "rLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwg7Yqc7YSw7JeQIOuztOyXrCDso7zqs6"
           "AsIOOAjOykgOu5hCDshYDjgI3snbQg66eM65OgIOydtOumhChkZiDCtyBGT1JNVUxB"
           "UyDCtyBvZmYp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ65286rOgIOu2hOuqhe2eiC"
           "DsoIHsnLzshLjsmpQuIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkOyEnCDs"
           "mZwg64Ks64qU7KeAIl19LCAi6rO87KCcIjogeyLssKjrn4kg66qo7ZiV7J2YIEFJQy"
           "Ao7KCQKSI6IFsi7LGE7KCQ7ZGc7JeQ7IScIOywqOufiSDrqqjtmJUg7ZaJ7J2YIEFJ"
           "QyDtlZjrgpjrpbwg44CM7J2066aEIDog7Iir7J6QIOygkOOAjSDstpzroKUg7ZWcIO"
           "2WieycvOuhnCDqs6jrnbwg7Lac66Cl7ZaI64qU7KeAIO2ZleyduO2VmOyEuOyalC4g"
           "6rCS7J20IDIg7KCQIOuCruqyjCDrgpjsmZTri6TrqbQg6rOE7IiYIOqwnOyImOyXkC"
           "DsoIjtjrjsnYQg7Y+s7ZWo7ZWY7KeAIOyViuydgCDqsoPsnbTqs6AsIOq4sOykgOyE"
           "oCDsqr3snLzroZwg7YGs6rKMIOyYrOudvOqwlOuLpOuptCDqt7gg7KCB7ZWp7JeQ66"
           "eMIOyYpO2UhOyFi+ydtCDruaDsp4Qg6rKD7J6F64uI64ukIiwgIuuCtCDsoJDsiJjq"
           "sIAg66Gc6re47Jqw64+E7JmAIOqzhOyImCDqsJzsiJgg7KSRIOyWtOuKkCDsqr3sl5"
           "DshJwg7Ja06riL64Ks64qU6rCAIl0sICLssKjrn4kg66qo7ZiV7J2YIOywqOuguSAx"
           "64WEIOyCrOqzoOycqOu5hCAo67CwKSI6IFsi7LCo65+JIOuqqO2YlSDqs4TsiJjtkZ"
           "zsl5DshJwg7LCo66C5IO2VnCDtlonrp4wg6rOo6528IOyngOyImOulvCDslIzsmrQg"
           "6rCS7J2EIOOAjOydtOumhCA6IOyIq+yekCDrsLDjgI0g7Lac66ClIO2VnCDtlonsnL"
           "zroZwg7Lac66Cl7ZWY66m0IOuQqeuLiOuLpC4g7J2M7IiYKOyVvSDiiJIwLjAzKeqw"
           "gCDrgpjsmZTri6TrqbQg66Gc6re4IOyKpOy8gOydvOydmCDqs4TsiJjrpbwg6re464"
           "yA66GcIOy2nOugpe2VnCDqsoPsnbTqs6AsIDEg7J2EIOuEmOyXiOuLpOuptCDrtoDt"
           "mLjrpbwg67CY64yA66GcIOydveydgCDqsoPsnoXri4jri6QiLCAi7J20IOqwkuydtC"
           "AxIOuztOuLpCDtgbDqsIAg7J6R7J2A6rCALCDqt7jrpqzqs6Ag6re46rKD7J20IOya"
           "lOycqCDrrLjsnqXsnLzroZzripQg66y07IqoIOucu+yduOqwgCJdLCAi6rKs67O4IO"
           "qzhOyVvSDshLgg6rG0IOyYiOy4oSDsmpTsnKjsnZgg7LWc64yAwrfstZzshowg67mE"
           "ICjrsLApIjogWyLjgIzspIDruYQg7IWA44CN7J20IOunjOuTpOyWtCDrkZQg6rKs67"
           "O4IOyEuCDqsbTsl5Ag7LCo65+JIOuqqO2YleydmCDsmIjsuKHsnYQg6rG46rOgLCDq"
           "sIDsnqUg64aS7J2AIOqwkuydhCDqsIDsnqUg64Ku7J2AIOqwkuycvOuhnCDrgpjriI"
           "gg67Cw7IiY66W8IOOAjOydtOumhCA6IOyIq+yekCDrsLDjgI0g7Lac66ClIO2VnCDt"
           "lonsnLzroZwg7Lac66Cl7ZWY66m0IOuQqeuLiOuLpC4gMSDsl5Ag6rCA6rmd6rKMIO"
           "uCmOyZlOuLpOuptCDshLgg6rG07JeQIOqwmeydgCDqsJLsnbQg66ek6rKo7KeEIOqy"
           "g+ydtOqzoCjruIzrnpzrk5zCt+ywqOugueydtCDslYgg65Ok7Ja06rCU7Iq164uI64"
           "ukKSwg67qE7IWI7Jy866GcIOuDiOuLpOuptCDrsLDsiJjqsIAg7JWE64uI6528IOyw"
           "qOydtOyeheuLiOuLpCIsICLsnbQg7Y+t7J20IOustOyXh+qzvCDrrLTsl4fsnZgg67"
           "mE7J206rOgLCDqt7gg7Y+t7J20IOyZnCDssqsg7ZmU66m0IOqyrOyggeydmCDtjJDr"
           "i6gg6re86rGw6rCAIOuQmOuKlOqwgCJdLCAicGxvdCI6IFsi6rCA66Gc7LaV7J20ID"
           "B+MTAwIOuwseu2hOychOulvCDtj6ztlajtlZjripTsp4Agwrcg6rOh7ISg7J20IOuR"
           "mOyduOyngCjssKjrn4kg66qo7ZiVIO2VmOuCmCwg7ZW17IusIOuqqO2YlSDtlZjrgp"
           "gpIMK3IOyEuOuhnOy2leydtCDroZzqt7gg64iI6riI7J247KeAIMK3IOuRkCDstpXs"
           "l5Ag652867Ko7J2EIOuLrOyVmOuKlOyngCDssKjroYDroZwg67O07IS47JqULiDqs6"
           "HshKDsnbQg7ZWY64KY67+Q7J2066m0IOuRkCDrqqjtmJUg7KSRIO2VmOuCmOunjCDs"
           "mIjsuKHtlZwg6rKD7J206rOgLCDshLjroZzstpUg6rCS7J20IDAuMSDrs7Tri6Qg7Y"
           "Gs6rKMIOuCruyVhOyngOuptCDrhbjstpzsnbQg6rOx7ZW07KeEIOqxtOyImOulvCDq"
           "t7jrprAg6rKD7J6F64uI64ukIiwgIuuCtCDqt7jrprzsnZgg6rOh7ISgIOyImOyZgC"
           "DshLjroZzstpUg64iI6riI7J20IOq4sOykgOqzvCDrp57sp4Ag7JWK64qU7KeAIl0s"
           "ICJlcnJvciI6IFsi7Jik66WYIOusuOyepSDssqsg7ZaJ66eMIOuWvOyWtCDsnb3snL"
           "zshLjsmpQg4oCUIOydtOumhOydhCDsnpjrqrsg7KCB7JeI64qU7KeAKE5hbWVFcnJv"
           "ciksIO2RnOyXkCDsl4bripQg7Je07J2EIOu2iOuggOuKlOyngChLZXlFcnJvcikg6r"
           "CAIOq3uCDtlonsl5Ag7KCB7ZiAIOyeiOyKteuLiOuLpC4g44CM7KSA67mEIOyFgOOA"
           "jeydtCDrp4zrk6Ag7J2066aEKGRmIMK3IENBTkRJREFURVMgwrcgRl9WRUggwrcgb2"
           "ZmIMK3IFNBTVBMRTMp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag7JWK7Iq164uI64ukIiwg"
           "IuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19fQ"
           "=="
)
_ASK = {"m1": "full", "m2": "blank",       # 뒤로 갈수록 걷힙니다 — 미션 3 은 항목 틀만 남습니다
        "m3": "self", "과제": "bare"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈 곳은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 항목 틀 (문장으로 완성하는 것은 학생 몫입니다)",
             "self": "◼ 스스로 점검할 항목 틀 (힌트 한 줄 뒤부터는 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 시작 행 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 시작 행 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 시작 행 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_STEP = {}                                 # 미션마다 힌트 계단을 몇 단계까지 열었는지
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 줄만 고릅니다 — 단위가 붙은 줄, 또는 「이름 = 값」·「이름 : 값」 꼴의 줄.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 이름과 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 이름을 붙여 고른 줄이 하나라도 있으면 그
    줄들만 보고, 그런 줄이 아예 없을 때만 출력 전체를 봅니다. 로그우도처럼 단위가 없는 값도
    「이름 : 숫자」 로 찍히므로 쌍점도 이름 자리로 함께 셉니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or ":" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.hlines(...)` 로 화면에 분명히 그어 놓은 가로선이 「가로선이 아예 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 셋으로 갈라 돌려줍니다 — 곡선(점 10개 이상) · 가로 기준선 · 세로 기준선.

    세로축 값이 처음부터 끝까지 같은 선은 점이 몇 개든 **가로 기준선** 으로 셉니다. `axhline` 으로 긋든
    같은 값을 열 번 늘어놓아 긋든 그림이 나타내는 것은 「상수 하나짜리 수평선」 으로 같은데, 뒤엣것을
    곡선으로 세면 곡선 개수 기준이 그림 그리는 방법 차이로 달라지거든요.
    """
    cur, hline, vline = [], [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 오류를 내도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.hlines` · `plt.vlines` 로 그은 선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 2 and yd.size == xd.size and np.allclose(yd, yd[0]):
            hline.append(float(yd[0]))
        elif xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, hline, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 분포한 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 분포한 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 맞지 않는 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, hline, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개 —"
                        " 상수 하나인 가로선은 곡선으로 세지 않습니다)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.4g}~{xd.max():.4g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 기준이 재는 자리는 가장 높은 점 하나입니다
                floor = float(xd[yd.argmin()])           # 그 자리에 가장 낮은 점이 서면 방향을 의심할 자리
                if not (p0 <= peak <= p1):
                    if p0 <= floor <= p1:
                        gaps.append(f"곡선의 최솟값 위치가 기준 구간 안의 {floor:.4g} 다 (최댓값이 와야 할 구간입니다."
                                    " 가로축 순서나 세로축 값이 뒤집혀 있는지 확인합니다)")
                    else:
                        gaps.append(f"곡선의 최댓값 위치가 {peak:.4g} 다 (기준 {p0:g}~{p1:g})")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "hline_y" in spec:
            h0, h1 = spec["hline_y"]
            if not any(h0 <= v <= h1 for v in hline):
                gaps.append("모형이 매기는 단일 요율을 나타내는 가로선이 기준 구간에 없다"
                            + (f" (지금 {hline[0]:.4g})" if hline else " (가로선이 아예 없다)"))
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("기준 구간 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4g})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
        if heights and "ymin" in spec and not (spec["ymin"][0] <= min(heights) <= spec["ymin"][1]):
            gaps.append(f"가장 낮은 막대가 {min(heights):,.2f} 다"
                        f" (기준 {spec['ymin'][0]:,.0f}~{spec['ymin'][1]:,.0f} —"
                        " 세로축에 AIC 말고 BIC 나 -2ℓ 를 올렸는지 봅니다)")
        if heights and "argmax" in spec:
            label = [t.get_text() for t in ax.get_xticklabels()]
            hi_cell = label[int(np.argmax(heights))] if len(label) == len(heights) else ""
            if hi_cell != spec["argmax"]:
                gaps.append(f"가장 높은 막대가 「{hi_cell}」 다 (기준 「{spec['argmax']}」)")
    elif spec.get("kind") == "barh":
        length = [float(p.get_width()) for p in ax.patches]      # 가로 막대는 길이가 값입니다
        if "n" in spec and len(length) != spec["n"]:
            gaps.append(f"막대가 {len(length)}개다 (기준 {spec['n']}개)")
        if length and "xmax" in spec and not (spec["xmax"][0] <= max(length) <= spec["xmax"][1]):
            gaps.append(f"가장 긴 막대가 {max(length):.4f} 다"
                        f" (기준 {spec['xmax'][0]:g}~{spec['xmax'][1]:g})")
        if "vline_x" in spec:
            _, _, vlines = _lines(ax)
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vlines):
                gaps.append("「영향 없음」인 배율 1 을 가리키는 세로선이 없다"
                            + (f" (지금 {vlines[0]:.4g})" if vlines else " (세로선이 아예 없다)"))
        if spec.get("xlabel") and not ax.get_xlabel().strip():
            gaps.append("가로축 라벨이 비어 있다 (무엇의 배율인지 축에 적습니다)")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 모음(LineCollection)은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 구간 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 분포한 구간에 없다")
    if spec.get("yscale") and ax.get_yscale() != spec["yscale"]:
        gaps.append(f"세로축이 {ax.get_yscale()} 눈금이다 (기준 {spec['yscale']} 눈금 —"
                    " 보통 눈금이면 낮은 구간이 바닥에 눌려 기울기를 읽을 수 없다)")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 「준비 셀」의 bm_x · bm_rate 로 관측 꺾은선을 그리고 단일 요율을 가로선으로 얹습니다"""
    x_, r_, c_, e_ = _get("bm_x"), _get("bm_rate"), _get("CLAIMS_ALL"), _get("EXP_ALL")
    if x_ is None or r_ is None or not c_ or not e_:
        return None
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.plot(x_, r_, "o-", color=COL["하늘색"], lw=2)
    a.axhline(float(c_) / float(e_), color=COL["남색"], ls=":")
    a.set_yscale("log")
    a.set_xlabel("할증점수 (구간 중앙값)")
    a.set_ylabel("연간 사고율 (건/년, 로그 눈금)")
    return f


_ref_cache = {}                                # 참조 그림이 한 번만 적합하도록 담아 두는 자리


def _ref_plot_m2():
    """미션 2 의 참조 그림 — 「준비 셀」의 df · F_CORE · off 로 계수별 사고율비 가로 막대를 그립니다"""
    d, formula, o_ = _get("df"), _get("F_CORE"), _get("off")
    if d is None or not formula or o_ is None:
        return None
    ratio = _ref_cache.get("m2")
    if ratio is None:
        fit = smf.glm(formula=formula, data=d, family=sm.families.Poisson(), offset=o_).fit()
        ratio = np.exp(fit.params.drop("Intercept"))     # 절편은 기준점이라 뺍니다
        _ref_cache["m2"] = ratio
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.6))
    a = f.add_subplot(111)
    bars = a.barh(list(ratio.index), ratio.to_numpy(), color=COL["연회색"])
    for one, name in zip(bars, ratio.index):
        if name == "BonusMalus":
            one.set_color(COL["하늘색"])
    a.axvline(1.0, color=COL["남색"], ls=":")
    a.set_xlim(0.97, 1.53)
    a.set_xlabel("사고율비 — 사고율에 곱해지는 배율")
    return f


def _ref_plot_m3():
    """미션 3 의 참조 그림 — 「준비 셀」의 df · FORMULAS · off 로 후보별 AIC 세로 막대를 그립니다"""
    d, cands, o_ = _get("df"), _get("FORMULAS"), _get("off")
    if d is None or not cands or o_ is None:
        return None
    scores = _ref_cache.get("m3")
    if scores is None:                                  # 세 적합은 한 번만 — 두 번째부터는 담아 둔 값을 씁니다
        scores = {}
        for name, formula in cands.items():
            fit = smf.glm(formula=formula, data=d, family=sm.families.Poisson(), offset=o_).fit()
            scores[name] = -2 * float(fit.llf) + 2 * (int(fit.df_model) + 1)
        _ref_cache["m3"] = scores
    names = [n for n in ("null", "core", "full") if n in scores]
    vals = [scores[n] for n in names]
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    bars = a.bar(names, vals, color=COL["연회색"])
    bars[int(np.argmin(vals))].set_color(COL["하늘색"])    # 가장 낮은 막대 하나만 강조
    a.set_ylim(285000, 295000)                        # 보이는 범위만 자릅니다 — 막대 높이는 그대로입니다
    a.set_xlabel("후보 모형")
    a.set_ylabel("AIC (점, 낮을수록 좋다)")
    return f


def _ref_plot_task():
    """과제의 참조 그림 — 「준비 셀」의 df · CANDIDATES · off 로 두 모형의 예측 요율 곡선을 그립니다"""
    d, cands, o_ = _get("df"), _get("CANDIDATES"), _get("off")
    if d is None or not cands or o_ is None or not {"vehicle", "core"} <= set(cands):
        return None
    curve = _ref_cache.get("과제")
    if curve is None:                                  # 두 적합은 한 번만 — 두 번째부터는 담아 둔 값을 씁니다
        curve = {}
        for name in ("vehicle", "core"):
            fit = smf.glm(formula=cands[name], data=d, family=sm.families.Poisson(), offset=o_).fit()
            rate = np.asarray(fit.predict(d))         # 오프셋 없이 = 노출 1년 기준의 연간 사고율
            curve[name] = np.percentile(rate, np.arange(0, 101))
        _ref_cache["과제"] = curve
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.plot(np.arange(0, 101), curve["vehicle"], color=COL["하늘색"], lw=2, label="차량 모형")
    a.plot(np.arange(0, 101), curve["core"], color=COL["남색"], lw=2, label="핵심 모형")
    a.set_yscale("log")
    a.set_xlabel("계약을 예측 요율 순으로 늘어놓은 백분위 (%)")
    a.set_ylabel("예측 연간 사고율 (건/년, 노출 1년 기준, 로그 눈금)")
    a.legend()
    return f


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2,   # 미션마다 참조 그림을 그리는 함수 하나
        "m3": _ref_plot_m3, "과제": _ref_plot_task}


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 줄 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_STUB = {"", "목표 —", "쓸 변수 —", "출력 —", "목표 · 쓸 변수 · 출력",
                "여기에 내 말로 적습니다", "내 말로 적습니다"}


_PROMPT_TAIL = " — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다"


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if txt in _PROMPT_STUB:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    if len(txt) < 30:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 부분이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                    # PROMPT 없이 통과하는 길은 여기서 막습니다
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason}{_PROMPT_TAIL}")
        print("   채점 결과와 풀이는 PROMPT 입력란을 채우고 이 셀을 다시 실행하면 열립니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    failed, passed = [], True
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 PROMPT 의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 이름(「:」·「=」)·단위가 붙은 줄이 있으면 그 줄만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) >= 10       # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    #  문턱이 10 인 까닭 — 채점표는 세 행만 찍어도 숫자가 18개라 30 으로는 못 걸렀습니다.
    #  이름(「:」·「=」)이나 단위가 붙은 줄이 하나라도 있으면 그 줄만 보므로(_값줄),
    #  값 두 줄을 제대로 찍고 표를 함께 띄운 정상 풀이는 이 문턱에 걸리지 않습니다.
    if table_flood:
        print(f"❌ 값 — 이름도 단위도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를 통째로 띄우면"
              " 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 : 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        hits = [v for v in (_nums("\n".join(tail)) if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            tail = (" " + unit) if unit else ""      # 로그우도처럼 단위가 없는 값은 빈 자리를 남기지 않습니다
            print(f"❌ {label} — 「{label} : 숫자{tail}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _STEP.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 그림 안 축이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    if failed:
        _STEP[key] = _STEP.get(key, 0) + 1   # ❌ 가 거듭되면 계단을 한 단계 올립니다
    for name, hkey in failed:
        first_tier = tbl.get(hkey, ("",))[0]
        first_tier = list(first_tier) if isinstance(first_tier, (tuple, list)) else [first_tier]
        shown_n = max(1, min(_STEP.get(key, 1), len(first_tier)))
        for line in first_tier[:shown_n]:
            if line:
                print(f"   힌트({name}) : {line}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 줄이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 준비 셀(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 원래대로 복원하고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면"
          " 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 절편만 둔 기준선 모형과 어제 사고율의 일치

### 미션 1 — 어제의 사고율을 재현하는 기준선 모형

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과(출력)에 찍히는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문** — 설명변수를 하나도 넣지 않은 모형을 적합하면 어제 구한 그 사고율이 그대로 재현됩니까?

그런 모형을 **null 모형**이라고 부릅니다. 우변에 절편만 남아 로그 스케일의 상수 하나가 됩니다. 노출의 로그는 계수 1 로 고정된 채 우변에 더해지는데, 이 항이 **오프셋**입니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 계약 678,013행 표, 사고 건수 열은 `ClaimNb` · 노출 열은 `Exposure` | 고정 |
| `off` | 노출의 로그, 오프셋에 넘길 값 (노출 1년이면 0) | 고정 |
| `CLAIMS_ALL` | 총 사고 건수, 단위 건 (채점 뒤 완성본의 총 사고 ÷ 총 노출 과 대조할 값) | 고정 |
| `EXP_ALL` | 총 노출, 단위 년 (채점 뒤 완성본의 총 사고 ÷ 총 노출 과 대조할 값) | 고정 |
| `bm_x` | 할증점수 구간 중앙값 10개, 단위 점 | 고정 |
| `bm_rate` | 구간별 관측 연간 사고율 10개, 단위 건/년 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **연간 사고율** — 단위 건/년 · 소수점 아래 6자리 · 절편만 둔 포아송 GLM 의 절편을 지수를 취해 복원한 값
2. **null 모형의 로그우도** — 단위 없음 · 소수점 아래 2자리 · 그 적합이 최대화한 값
   - 어제 최소화한 음의 로그우도에 −1 을 곱한 값이 여기 나옵니다
3. **함께 출력할 한 행** — 로그 스케일 그대로의 절편
   - 채점 기준에 들어가는 값은 아니고, 통과 뒤 열리는 완성본과 대조할 항목입니다
4. **그래프 한 장** — 가로축은 할증점수 구간 중앙값 52.5 ～ 190 점, 세로축은 연간 사고율인 꺾은선 그래프
   - 꺾은선 : `bm_x` 와 `bm_rate` 를 이은 선 하나. 범례 문구는 「구간별 관측 사고율」
   - 가로선 : 1번에서 구한 연간 사고율 값에 맞춰 가로축 전체를 가로지르는 선 하나. 범례 문구는 「null 모형이 매기는 단일 요율」
     - 이 가로선은 곡선 개수에 포함하지 않으므로 꺾은선은 그대로 1개입니다
   - 세로축 눈금 : 로그 눈금
   - 가로축 이름 : 「할증점수 (구간 중앙값)」
   - 세로축 이름 : 「연간 사고율 (건/년, 로그 눈금)」

**출력 형식** — 「이름 = 값 단위」 형식으로 한 행씩 `print` 합니다. 그래프는 한 장입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `off` · `bm_x` · `bm_rate` 는 이미 있으니 다시 만들지 말라고 적습니다.
- **모형 설정** — 우변에 설명변수를 하나도 두지 않는다는 것, 분포를 포아송으로 명시하는 것, 노출은 `off` 를 오프셋으로 넘긴다는 것.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지. 절편은 로그 스케일의 값이라 지수를 취해 복원해야 건/년이 됩니다.
- **그래프** — 가로축·세로축에 무엇을 놓을지 · 세로축 눈금 · 가로선이 무엇의 값인지 · 가로축·세로축 이름과 범례.

**막히면** — AI 튜터에게 「포아송 회귀에서 노출 보정을 넣으려면 어떤 인자에 무엇을 넘기나요」 처럼 물어보세요.

#### 실행 전 — 예상 선택

> **[예측] 노출을 오프셋으로 단 null 모형을 적합하면 절편은 어떻게 찍힐까요?**
>
> 1. 절편이 그대로 어제의 사고율로 찍힌다
> 2. 절편은 음수인데 지수를 씌우면 어제의 사고율이 된다
> 3. 절편도 지수도 어제의 사고율과 무관한 값이 나온다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

고른 이유를 한 문장 적어 두면 채점 결과와 대조할 기준이 생깁니다.

<details class="lms-extra"><summary>배경 — 첫 모형의 구성과 오프셋 항의 위치</summary>

어제 노출 합으로 나눠 구한 사고율이 오늘 이 모형이 재현해야 할 기준값입니다. 우변에 절편 하나만 둔 GLM 이라 모든 계약에 같은 값을 부여합니다.

$$
\log(\mathbb{E}[y_i]) = \beta_0 + \log(E_i)
$$

- $y_i$ — 계약 $i$ 의 사고 건수 (단위 건). 모형이 예측하는 것은 요율이 아니라 이 **건수**입니다
- $E_i$ — 계약 $i$ 의 노출 (단위 년, 계약이 보장을 받은 기간)
- $\mu_i = \mathbb{E}[y_i] = E_i\,\lambda_i$ — 계약 $i$ 의 **기대 사고 건수** (단위 건). 연간 사고율 $\lambda_i$ 에 노출 $E_i$ 를 곱한 값입니다. 로그를 취하면 $\log \mu_i = \log E_i + \beta_0$ 이라 계수 없이 더해지는 $\log E_i$ 가 **오프셋** 항입니다 (이론 4. 포아송 회귀)
- $\beta_0$ — 로그 스케일의 절편. 요율표에 적을 연간 사고율 $\lambda$ 는 $\exp(\beta_0)$ 이고 단위는 건/년입니다

</details>

미션 1 이 그릴 null 모형이 어떤 모양인지 먼저 봅니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_003.png" width="640" height="427"/>
<p align="center"><em>▲ 모형 여러 개의 예측이 관측과 얼마나 맞는지 비교한 그림(이론 3. 로지스틱 회귀의 확률 보정을 건수 예측에 옮긴 것) — 오른쪽 아래 DummyRegressor 그래프가 오늘 만들 null 모형이다. 예측(파랑 x)과 관측(주황 원)이 어느 구간에서나 0.10 근처에서 평평하게 이어진다. 오른쪽으로 올라가는 곡선을 만드는 것이 다음 미션의 일이다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>주의 — AI 튜터에 맡길 작업과 직접 확인할 항목</summary>

「이 셀에 넣기」는 Claude 도우미가 연결돼 있어야 눌립니다(미연결이면 비활성). AI 튜터가 코드를 넣는 위치는 마지막으로 클릭한 코드 셀이라, 다른 셀을 클릭한 채 시키면 엉뚱한 셀이 바뀝니다.

왼쪽은 문법과 코드 구조 구성이라 맡겨도 되는 작업이고, 오른쪽은 **틀려도 오류가 나지 않아** 사람이 직접 확인해야 하는 작업입니다.

| AI 튜터에 맡길 작업 | 내가 반드시 확인할 항목 |
|---|---|
| `smf.glm` 호출문과 인자 채우기 | `family` 를 포아송으로 명시했는가 (적지 않으면 정규분포로 적합된다) |
| 계수표에서 절편을 추출해 인쇄 | `offset` 에 노출의 로그가 들어갔는가 (빠지면 분모가 계약 수가 된다) |
| 꺾은선과 가로선을 그리고 가로축·세로축 이름 달기 | 절편을 그대로 읽지 않고 **지수로 변환했는가**, 값이 어제 구한 값과 일치하는가 |

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
from sklearn.datasets import fetch_openml

# freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도 (678,013 x 12). OpenML 에서 바로 내려받습니다
df = fetch_openml(data_id=41214, as_frame=True).frame
df["IDpol"] = df["IDpol"].astype(int)
for c in ["Area", "VehBrand", "VehGas", "Region"]:
    df[c] = df[c].astype(str).str.strip("'").astype("category")

# 첫날 정해 닷새 내내 고정하는 전처리 두 규칙
df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)   # 사고 5건 이상은 극소수라 상한 4 로 상한 처리
df["Exposure"] = df["Exposure"].clip(upper=1.0)           # 노출은 1년 상한

off = np.log(df["Exposure"])                  # 쓸 변수 1 — 노출의 로그 (offset 에 넘길 값, 노출 1년이면 0)
CLAIMS_ALL = int(df["ClaimNb"].sum())         # 쓸 변수 2 — 총 사고 건수 (단위 건)
EXP_ALL = float(df["Exposure"].sum())         # 쓸 변수 3 — 총 노출 (단위 년)

bm_edges = [50, 55, 60, 70, 80, 90, 100, 110, 125, 150, 230]        # 쓸 변수 4 — 할증점수 구간 경계 (점)
_bm_cut = pd.cut(df["BonusMalus"], bm_edges, right=False)
_bm_sum = df.groupby(_bm_cut, observed=True)[["ClaimNb", "Exposure"]].sum()
bm_x = [(a + b) / 2 for a, b in zip(bm_edges[:-1], bm_edges[1:])]   # 쓸 변수 5 — 구간 중앙값 (그림의 가로축)
bm_rate = (_bm_sum["ClaimNb"] / _bm_sum["Exposure"]).to_numpy()     # 쓸 변수 6 — 구간별 관측 연간 사고율 (건/년)

print("행 x 열 :", df.shape, "| 결측치 총합 :", int(df.isna().sum().sum()))
print(f"총 사고 {CLAIMS_ALL:,} 건 | 총 노출 {EXP_ALL:,.1f} 년 |"
      f" log(Exposure) {off.min():.3f} ~ {off.max():.3f}  (노출 1년이면 0)")
print(f"할증점수 구간 {len(bm_x)}개 | 구간별 관측 사고율 {bm_rate.min():.4f} ~ {bm_rate.max():.4f} 건/년")
print("쓸 수 있는 이름 : df · off (노출의 로그) · CLAIMS_ALL (건) · EXP_ALL (년) ·"
      " bm_x (점) · bm_rate (건/년)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] AI 코드 시작 행 아래 코드에서 `offset=off` 인자만 지우고 그대로 다시 실행하면 절편의 지수는 어떻게 되고, 그 인자는 코드에서 무슨 일을 하고 있었나요?**
>
> 1. 0.100614 에서 0.0532 로 내려간다 — `offset` 이 각 계약의 노출을 로그 스케일로 우변에 더해 주는 항이라, 빠지면 분모가 노출의 합이 아니라 계약 수가 된다
> 2. 0.100614 에서 0.0532 로 내려간다 — `offset` 이 사고 건수를 노출로 나눠 타깃 자체를 요율로 바꿔 주는 인자라, 빠지면 타깃이 건수로 돌아간다
> 3. 값은 그대로다 — `offset` 은 계수표의 이름만 바꾸는 인자다

## 계약 특성을 넣은 모형의 계수 해석

### 미션 2 — 계수의 요율 문장 변환 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 **먼저 그대로 ▶ 해서** 전임 담당자 코드의 채점 결과(❌)를 봅니다.
3. 아래 고칠 것대로 「PROMPT 셀」의 PROMPT 를 고치고, 미션 1 의 3번부터 같은 순서로 진행합니다.

**이 미션의 질문** — 로그 스케일의 계수 8개를 요율표에 적을 배율로 옮기려면 어디를 고쳐야 합니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 계약 678,013행 표 | 고정 |
| `F_CORE` | 핵심 모형의 formula 문자열, 우변은 할증점수·나이·도시화 등급·연료 | 고정 |
| `off` | 노출의 로그, 오프셋에 넘길 값 (노출 1년이면 0) | 고정 |
| `CLAIMS_ALL` | 총 사고 건수, 단위 건 (예측 합과 대조할 값) | 고정 |

#### 고칠 것

AI 코드 시작 행 아래에 전임 담당자 코드가 이미 들어 있습니다. 오류 없이 돌아가는데 값이 틀렸고 그림이 없습니다. 고친 뒤의 아래 4개가 이 미션의 제출물입니다.

1. **할증점수 1점의 사고율비** — 단위 배 · 소수점 아래 6자리 · 로그 스케일 계수 `BonusMalus` 에 지수를 씌운 값
   - 지금 : 배율이 아닌 값이 출력됩니다
   - 고친 뒤 : 로그 스케일 계수에 지수를 한 번 씌운 값을 출력합니다 — 지금 코드는 계수에 100 을 곱해 백분율로 만든 뒤 다시 1 을 더하는 1차 근사를 사용합니다
   - 이 배율은 50～230점 전체에 하나뿐이라 100점 부근의 꺾임을 담지 못합니다. 미션 1 그림에서 95점 구간 0.1514 건/년이 105점 구간 0.2810 건/년으로 뛴 두 점이 그 꺾임이고, 실제 요율표에서는 할증점수도 등급 구간으로 나눠 적습니다
2. **도시화 등급 F 의 사고율비** — 단위 배 · 소수점 아래 6자리 · 기준 등급 A 대비 · 로그 스케일 계수 `C(Area)[T.F]` 에 지수를 씌운 값
   - 지금 : 값이 0.08 배쯤 낮게 출력됩니다
   - 고친 뒤 : 1번과 같은 행을 고치면 이 값도 함께 정정됩니다 — 계수가 커도 규칙은 같은 지수 하나입니다
3. **그래프 한 장** — 가로축은 사고율비, 세로축은 계수 이름인 가로 막대 그래프
   - 막대 : 절편을 뺀 계수 하나에 막대 하나, 모두 8개
   - 세로선 : 「영향 없음」에 해당하는 배율 1.0 에 세로축 전체를 가로지르는 선 하나
     - 이 세로선은 막대 개수에 포함하지 않으므로 막대는 그대로 8개입니다
   - 가로축 이름 : 「사고율비 — 사고율에 곱해지는 배율」
   - 지금 : 그림이 없습니다. 같은 프롬프트에서 새로 시켜야 나옵니다
4. **남길 표 한 장** — 로그 스케일의 계수와 사고율비를 나란히 놓은 표를 그대로 남겨 둡니다
   - 채점 기준에 들어가는 값은 아니고, 통과 뒤 열리는 완성본과 대조할 항목입니다

**통과 기준** — 값 2개가 「이름 = 값 단위」 형식으로 한 행씩 출력되고, 가로 막대 한 장이 위 요구를 다 갖추면 통과입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Generic_forest_plot.png" width="620" height="401"/>
<p align="center"><em>▲ 세로축에 항목 이름을 늘어놓고 가로축에 배율을 놓은 의학 논문용 그림(<strong>숲 그림</strong>, forest plot) — 1.0 에 그은 세로선이 「영향 없음」을 나타내고, 그 선의 오른쪽은 위험을 올리는 항목, 왼쪽은 내리는 항목이다. 우리가 미션 2 에서 그릴 것이 이 배치다 (출처: Wikimedia Commons, James Grellier, CC BY-SA 3.0)</em></p>

<details class="lms-extra"><summary>자세히 — 숲 그림과의 공통점과 차이점</summary>

여기 쓰인 OR(odds ratio, 오즈비 — 사건이 일어날 확률을 안 일어날 확률로 나눈 값들의 비)은 우리의 사고율비와 다른 지표이고, 검은 사각형과 가로 선분도 오늘 그릴 막대와는 다릅니다. 참고할 것은 배치 하나입니다.

</details>

#### 프롬프트에 넣을 것(힌트)

- **고칠 행** — 계수를 배율로 변환하는 코드 행이 어디이고 무엇으로 바뀌어야 하는지, 백분율 변화는 그 배율에서 어떻게 만드는지.
- **손대지 말 범위** — 적합 코드 행과 `beta` 를 만드는 행은 그대로 두라고 적습니다. `df` · `F_CORE` · `off` 도 다시 만들지 않습니다.
- **출력** — 값 2개의 이름과 단위, 골라 출력할 계수 이름 2개, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축·세로축에 무엇을 놓을지 · 막대 개수 · 세로선의 위치 · 가로축 이름.

**막히면** — AI 튜터에게 「로그 스케일의 계수를 원래 단위의 배율로 변환하는 규칙이 무엇인가요」 처럼 물어보세요.

#### 고치기 전 — 틀린 코드 행의 확인

AI 코드 시작 행 아래 4개 행을 한 행씩 읽고 변환이 일어나는 행을 확인하세요.

> **[예측] 「PROMPT 셀」의 AI 코드 시작 행 아래 코드는 에러 없이 돌지만 답이 틀렸습니다. 틀린 곳은 어디이고 왜일까요?**
>
> 1. `beta = res_core.params.drop("Intercept")` — 절편을 빼서 기준점이 사라졌다
> 2. `rr = 1 + pct / 100` — 로그 스케일의 계수를 「1 + β」 로 근사해 배율인 척 만들었다
> 3. `print(f"… {float(rr['BonusMalus']):.6f} 배")` — 소수점 아래 여섯째 자리까지 잘라 틀린 값이 나왔다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

고른 쪽과 이유를 적어 두세요.

<details class="lms-extra"><summary>배경 — 우변에 넣을 특성 4개와 배율 변환 규칙</summary>

미션 1 의 기준선 모형은 계약 67만 건 모두에게 같은 값을 부여했습니다. 「준비 셀」이 아래 4개를 담은 formula 를 만들어 둡니다 — 오늘 「핵심 모형」이라고 부를 모형입니다.

| 특성 | 컬럼 | 단위 |
|---|---|---|
| 할증점수 | `BonusMalus` | 점 |
| 나이 | `DrivAge` | 살 |
| 도시화 등급 | `Area` | 등급 A～F |
| 연료 | `VehGas` | 경유·휘발유 |

$$
\log(\lambda_i) = \beta_0 + \beta_{\text{BM}}\,\text{BonusMalus}_i + \beta_{\text{age}}\,\text{DrivAge}_i + (\text{Area 더미}) + (\text{VehGas 더미})
$$

- $\lambda_i$ — 계약 $i$ 의 연간 사고율 (단위 건/년)
- $\mu_i = E_i\,\lambda_i$ — 계약 $i$ 의 **기대 사고 건수** (단위 건). 로그를 취하면 계수 없이 더해지는 $\log E_i$ 가 미션 1 과 같은 오프셋 항입니다
- $\beta_{\text{BM}}$ — 할증점수가 1점 높을 때 **로그 사고율에 더해지는 값**. 로그 스케일의 값이라 이대로는 요율표에 못 적습니다
- **Area 더미** — `C()` 로 감싼 변수는 범주형이라 statsmodels 가 더미로 전개합니다. 등급이 6개면 기준 A 를 뺀 5개가 생성됩니다 (이론 5. AIC 와 BIC). 계수 이름 `C(Area)[T.F]` 는 「기준 A 대비 F」 로 읽습니다. 이론에서는 같은 계수가 `Area[T.F]` 로 출력되는데, `C()` 로 감쌌는지만 다르고 값은 같습니다

로그 스케일에서 더해지던 계수는 원래 단위에서 곱해집니다. 양변에 지수를 씌우면 다음과 같습니다.

$$
\lambda_i = e^{\beta_0} \cdot \left(e^{\beta_{\text{BM}}}\right)^{\text{BonusMalus}_i} \cdots
$$

1단위 오를 때 사고율에 곱해지는 $e^{\beta}$ 가 **사고율비**이고(이론 4. 포아송 회귀), 배율 1.0 이 「영향 없음」에 해당하는 값입니다(이론 6. 실제 데이터 적용 — 요인별 배율 그림의 세로선). 배율 1.05 면 1단위당 5% 상승이고, 백분율로 옮길 때는 배율에서 1을 빼고 100을 곱합니다.

계수가 아주 작을 때는 $e^{\beta}$ 와 $1 + \beta$ 가 거의 같습니다 — 계수 0.02 면 2.00% 와 2.02% 의 차이입니다. 그런데 계수 0.4 면 40.0% 와 실제 49.2% 로 차이가 커집니다. **역변환 규칙은 계수 크기와 상관없이 하나**입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Exp_tangent.svg" width="320" height="500"/>
<p align="center"><em>▲ 파란 곡선이 지수함수이고, 빨간 직선은 곡선에 한 점에서 닿는 접선이다(초록 삼각형은 그 접선의 기울기를 표시한 것). 접점 바로 옆에서는 두 선이 거의 일치해 직선으로 바꿔 읽어도 차이가 보이지 않지만, 접점에서 멀어질수록 곡선과 접선의 차이가 빠르게 커진다 — 계수가 작을 때만 통하는 근사가 큰 계수에서 차이가 커지는 모양이 이것이다 (출처: Wikimedia Commons, Dmcq, CC BY-SA 3.0)</em></p>

</details>

<details class="lms-extra"><summary>채점 뒤 확인 — 계수표에서 DrivAge 부호 보기</summary>

`DrivAge` 의 계수 부호를 확인하세요. 젊은 운전자의 위험이 이미 할증점수에 기록돼 있다면 부호가 예상과 다를 수 있습니다.

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 열어도 혼자 섭니다
    from sklearn.datasets import fetch_openml                    # (미션 1 을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 첫날 정한 전처리 두 규칙 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

F_CORE = "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"   # 쓸 변수 1 — 핵심 모형의 formula 문자열
off = np.log(df["Exposure"])                  # 쓸 변수 2 — 노출의 로그 (offset 에 넘길 값, 노출 1년이면 0)
CLAIMS_ALL = int(df["ClaimNb"].sum())         # 쓸 변수 3 — 총 사고 건수 (단위 건, 대조용)

print("핵심 모형 formula :", F_CORE)
print("범주 수준 — Area :", list(df["Area"].cat.categories),
      "| VehGas :", list(df["VehGas"].cat.categories))
print(f"계약 {len(df):,} 건 | 총 사고 {CLAIMS_ALL:,} 건"
      "  (절편이 있는 포아송 GLM 이면 예측 사고 건수의 합이 이 값과 맞습니다)")
print("쓸 수 있는 이름 : df · F_CORE (formula 문자열) · off (노출의 로그) · CLAIMS_ALL (건)")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 돌아가지만 답이 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 구분선 아래를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 그대로 실행하면 채점 결과가 ❌ 를 냅니다
res_core = smf.glm(formula=F_CORE, data=df, family=sm.families.Poisson(), offset=off).fit()
beta = res_core.params.drop("Intercept")     # 로그 스케일의 계수 8개 (절편은 기준점이라 뺍니다)
pct = beta * 100                             # 계수를 백분율 변화로
rr = 1 + pct / 100                           # 백분율 변화를 배율로

print(f"적합 — 관측 {int(res_core.nobs):,} 건 · 계수 {int(res_core.df_model) + 1} 개"
      f" · 예측 사고 건수 합 {res_core.fittedvalues.sum():,.0f} 건 (실적 {CLAIMS_ALL:,} 건)")
print(f"할증점수 1점의 사고율비 : {float(rr['BonusMalus']):.6f} 배")
print(f"도시화 등급 F 의 사고율비 : {float(rr['C(Area)[T.F]']):.6f} 배")
print(pd.DataFrame({"계수(로그)": beta, "사고율비": rr}).round(4).to_string())

> **[문제] AI 코드 시작 행 아래 코드의 `rr = 1 + pct / 100` 한 행이 하는 일은 무엇이고, 무엇으로 바꿔야 사고율비가 되나요?**
>
> 1. 계수를 백분율로 바꾸는 코드 행이다 — 바로 위 행 `pct = beta * 100` 을 지우면 값이 맞는다
> 2. 배율을 백분율로 복원하는 코드 행이다 — `beta` 대신 `res_core.params` 를 넣어 절편까지 포함시키면 값이 맞는다
> 3. 로그 스케일의 계수를 「1 + β」 로 근사해 배율인 척 만드는 코드 행이다 — `rr = np.exp(beta)` 로 바꿔야 계수 크기와 상관없이 배율이 된다

## 후보 3개의 비교와 요율표 선정

### 미션 3 — 후보 3개의 동일 조건 비교

#### 할 일 — 미션 3 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시키고, 미션 1 의 3번부터 같은 순서로 진행합니다.

**이 미션의 질문** — 미션 2 의 요율 문장에 변수를 더 넣으면 그 표가 더 좋아집니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 계약 678,013행 표 | 고정 |
| `FORMULAS` | 후보 3개의 이름과 formula, 키는 `null` · `core` · `full` | 고정 |
| `F_CORE` | 핵심 모형의 formula 문자열, 계수 9개 | 고정 |
| `F_FULL` | 확장 모형의 formula 문자열, 계수 41개 | 고정 |
| `off` | 노출의 로그, 오프셋에 넘길 값 (노출 1년이면 0) | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **가장 낮은 AIC** — 단위 점 · 소수점 아래 2자리 · 후보 3개의 AIC 중 가장 작은 값. 어느 모형인지도 같은 행에 적습니다
2. **채점표 null 행의 로그우도** — 단위 없음 · 소수점 아래 2자리 · 그 적합이 최대화한 값이라 음수로 출력됩니다
   - 미션 1 에서 출력한 기준선 값과 일치해야 합니다
3. **함께 출력할 채점표 한 장** — 한 행이 후보 하나이고, 열은 이름 · 계수 개수 · 로그우도 · 이탈도(deviance — 완벽히 맞히는 가상 모형과 내 모형 사이의 로그우도 격차를 2배 한 값, 패널티 항이 없어 변수를 넣을수록 내려갑니다) · AIC · BIC · 그 적합에 쓰인 관측 수
   - 채점 기준에 들어가는 값은 아니지만, 3개 행의 관측 수가 서로 같고 계약 수와도 같은지 여기서 확인합니다
4. **그래프 한 장** — 가로축은 후보 이름, 세로축은 AIC 인 세로 막대 그래프
   - 막대 : 후보 하나에 막대 하나, 모두 3개. 가로축 눈금 이름은 `null` · `core` · `full` 그대로 씁니다
   - 강조 : 가장 낮은 막대 하나만 색을 달리합니다
   - 세로축 눈금 : 세 막대의 차이가 보이도록 잘라도 됩니다
   - 가로축 이름 : 「후보 모형」
   - 세로축 이름 : 「AIC (점, 낮을수록 좋다)」 를 글자 그대로 씁니다

**출력 형식** — 값 2개를 「이름 = 값 단위」 형식으로 한 행씩 `print` 합니다. 그래프는 한 장입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `FORMULAS` · `off` 는 이미 있으니 다시 만들지 말라고 적습니다.
- **같은 조건** — 후보 3개를 한 반복문에서 같은 분포·같은 타깃·같은 오프셋으로 적합하라는 것, 그리고 채점표에 관측 수 열을 넣는 것.
- **출력** — 채점표의 열 목록, 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지. AIC·BIC 를 로그우도와 계수 개수로 어떻게 만드는지도 적습니다(계수 개수에 절편을 포함합니다).
- **그래프** — 가로축·세로축에 무엇을 놓을지 · 막대 3개의 순서 · 강조할 막대 하나 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「적합 결과에서 절편까지 포함한 계수 개수를 어떻게 꺼내나요」 처럼 물어보세요.

표와 그림을 얻으면 한 문장을 더 적어 두세요 — 「요율표 후보로 남길 것과 그 근거」. 정리에서 다시 씁니다.

#### 실행 전 — 예상 선택

계수가 1개 · 9개 · 41개이고 BIC 의 패널티 항이 AIC 의 6배가 넘는다는 점을 고려해 고른 뒤, 근거 한 문장을 추가하세요.

> **[예측] null · core · full 세 후보를 AIC 와 BIC 로 각각 순위를 매기면 1등이 어떻게 갈릴까요?**
>
> 1. 둘 다 full 이 1등
> 2. AIC 는 full, BIC 는 core 가 1등
> 3. AIC 는 full, BIC 는 null 이 1등
> 4. 둘 다 null 이 1등
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>배경 — 후보 3개와 순위 점수의 정의</summary>

후보는 3개입니다. 아무 특성도 안 넣은 기준선 `null`, 이론의 **핵심 모형**인 특성 4개의 `core`, 이론의 **확장 모형**으로 브랜드·지역·인구밀도를 추가해 계수가 41개가 된 `full` 입니다.

점수는 「이론 5. AIC 와 BIC」의 두 식 그대로입니다. 값의 크기에는 뜻이 없고 **같은 표 안 후보끼리의 차이**만 읽습니다.

$$
\mathrm{AIC} = -2\ell + 2k
$$

$$
\mathrm{BIC} = -2\ell + \log(n)\,k
$$

- $\ell$ 는 그 모형이 최대화한 **로그우도**, $k$ 는 절편까지 포함한 **계수 개수**, $n$ 은 그 적합에 **쓰인 관측 수** 입니다
- 후보 3개의 $k$ 는 1 · 9 · 41 이고, 「준비 셀」이 $\log n$ 을 찍어 줍니다
- $-2\ell$ — 로그우도에 −2 를 곱해 「낮을수록 좋다」 로 방향을 반전시킨 값. 뒤에 더해지는 것이 패널티 항입니다
- 계약 678,013건이면 $\log n = 13.43$ 이라, 계수 1개마다 AIC 는 2 를 · BIC 는 13.43 을 더합니다 — 계수를 32개 더 쓰면 패널티 항이 64 대 429.7 로 달라집니다

로그우도만으로는 멈출 곳을 못 정한다 — 「이론 5. AIC 와 BIC」에서 본 내용입니다. 계수를 늘려 이 표본의 잡음까지 적합하면 새 계약에서는 예측 오차가 오히려 커지는데, 그 상태가 **과적합**(overfitting)입니다.

「이론 5. AIC 와 BIC」의 표는 계약 2,000건을 추출해 계산한 것이라 그 표에서는 확장이 밀렸는데(핵심 752.7 < 확장 763.8), 우리는 계약 678,013건 전체로 계산합니다. 후보 3개의 AIC 는 「이론 5. AIC 와 BIC」 끝의 계약 678,013건 표에 적혀 있으니, 여러분 표의 값 3개가 소수점 아래 첫째 자리까지 그 3개 행과 맞아떨어지는지 대조해 보세요.

</details>

<details class="lms-extra"><summary>자세히 — 패널티 항이 필요한 이유</summary>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Overfitting.svg" width="420" height="420"/>
<p align="center"><em>▲ 같은 두 무리를 초록 선과 검은 선 경계 2개로 나눈 그림 — 가진 표본의 점에 완벽할수록 좋은 모형이 아니라는 말이 이 모양이다 (출처: Wikimedia Commons, Chabacano, CC BY-SA 4.0)</em></p>

**초록 선**은 큰 점을 하나도 틀리지 않으려고 여러 번 꺾였고, **검은 선**은 몇 개를 틀리는 대신 매끈합니다.

아래는 눈금 없는 개념도입니다. 가로축이 모형 복잡도(우리에게는 계수 개수 k), 세로축이 새 데이터에서의 오차입니다.

**빨강(Bias²)** 은 모형이 단순해서 나는 오차라 오른쪽으로 갈수록 줄어듭니다. **청록(Variance)** 은 표본의 잡음까지 적합해 나는 오차라 오른쪽으로 갈수록 증가합니다. 둘을 합친 **검정(Total Error)** 이 점선 위치에서 가장 낮아집니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Bias_and_variance_contributing_to_total_error.svg" width="620" height="390"/>
<p align="center"><em>▲ 복잡도가 오를수록 청록 Variance 가 커져 검정 Total Error 가 U 자를 그린 개념도 — AIC 의 <code>+2k</code> 는 오른쪽으로 갈수록 커지는 이 오차를 계수 개수로 어림잡아 점수에 넣는 패널티 항이다 (출처: Wikimedia Commons, Bigbossfarin, CC0)</em></p>

</details>

AIC 는 후보의 순위만 매깁니다. 그 후보가 계약을 실제로 얼마나 구분하는지는 이런 그림으로 따로 확인합니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_004.png" width="480" height="480"/>
<p align="center"><em>▲ 같은 데이터(freMTPL2freq)에서 모형이 계약을 위험 순으로 얼마나 잘 구분하는지 그린 <strong>로렌츠 곡선</strong> — 파랑 DummyRegressor 가 Gini(곡선이 대각선에서 처진 넓이를 0～1 로 계산한 수치) 0.01 로 대각선에 그대로 붙어, 모두에게 같은 값 하나를 매기는 기준선(null)이 위험을 하나도 구분하지 못한다는 것을 보여 준다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>자세히 — 로렌츠 곡선 읽는 법</summary>

계약을 모형이 매긴 요율의 안전한 순서대로 늘어놓고, 가로축에 노출의 누적 비율을, 세로축에 그 계약들이 실제로 낸 사고의 누적 비율을 표시한 곡선입니다. 검은 점선(Random baseline)이 대각선이고, 곡선이 아래로 깊이 처질수록 위험을 잘 구분한 것입니다.

여기 나온 4개는 오늘의 후보가 아니라 같은 데이터로 만든 다른 후보들이고, 참고할 것은 「특성이 적을수록 곡선이 대각선에 근접한다」 는 모양 하나입니다.

</details>

AIC 막대 한 장을 함께 그리도록 한 이유(「만들 것」 4번)가 아래 그림입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Anscombe%27s_quartet_3.svg" width="640" height="465"/>
<p align="center"><em>▲ 앤스컴 콰르텟 — 네 데이터의 평균·분산·상관계수·회귀직선(파란 선)이 전부 같은데, 주황 점의 모양은 흩어진 구름 · 휘어진 곡선 · 직선 하나에 이상점 하나 · 가로값이 8 에 몰린 세로 점 다발로 완전히 다르다. 점수 하나가 같다고 두 모형이 같은 일을 하는 것은 아니라는 뜻이고, 그래서 AIC 옆에 그림 한 장을 같이 놓는다 (출처: Wikipedia, Anscombe's quartet 3.svg)</em></p>

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 열어도 혼자 섭니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 첫날 정한 전처리 두 규칙 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

F_CORE = "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"        # 쓸 변수 1 — 핵심 모형 (미션 2 와 같은 식)
F_FULL = ("ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"       # 쓸 변수 2 — 확장 모형 (계수 41개)
          " + C(VehBrand) + C(Region) + np.log(Density)")
FORMULAS = {"null": "ClaimNb ~ 1", "core": F_CORE, "full": F_FULL}     # 쓸 변수 3 — 후보 3개 (이름 -> formula)
off = np.log(df["Exposure"])                  # 쓸 변수 4 — 노출의 로그 (offset 에 넘길 값, 노출 1년이면 0)

for name, formula in FORMULAS.items():
    print(f"{name:5s} : {formula}")
print(f"\n계약 {len(df):,} 건 | log(n) = {np.log(len(df)):.4f}"
      "  -> BIC 가 계수 1개마다 더하는 패널티 항 (AIC 는 2)")
print("쓸 수 있는 이름 :  df · FORMULAS · F_CORE · F_FULL · off")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 여러분이 만든 채점표에서 확장 모형(full)의 AIC 가 셋 중 가장 낮았으니, 이 모형이 내년 새 계약의 사고 건수도 가장 정확하게 예측한다.**
>
> 1. O (맞다)
> 2. X (틀리다)

## 차량 정보만 사용한 첫 화면 견적의 타당성

> **[과제] 과제 제출 — 첫 화면에 쓸 차량 모형 고르기**
>
> - **무엇을** 차량 정보만 쓰는 모형을 첫 화면 견적에 써도 되는지 답 한 문장과, 후보 4개 채점표·요율 그림·견본 3건 요율 표를 만듭니다.
> - **무엇으로** 오늘 쓴 계약 데이터 하나로 하고, 「준비 셀」이 후보 4개의 식과 견본 계약 3건을 만들어 둡니다.
> - **인정** 채점이 값 3개와 그래프를 통과시킨 노트북에 브리핑과 채널팀 회신을 붙인 것입니다.
> - **예** 견본 「나 · 5년 · 경유」(B1 · 차령 5 · 출력 6)의 예측 요율을 두 모형에서 건/년으로 적고, 「차령 1년 사고율비 = 0.○○○○ 배」 한 행을 냅니다.
> 
> - 내가 쓴 `PROMPT` 를 그대로 둔 채 실행해 값 3개와 그래프를 모두 통과시킨 상태로 낼 것
> - 채점표에 그 적합에 쓰인 관측 수 열을 넣고, 후보 4개의 관측 수가 같은지를 한 행으로 출력할 것
> - 차량 모형의 계수 2개 이상을 「1단위 오르면 몇 배」 형태의 배율 문장으로 옮길 것
> - 같은 우변 로지스틱 회귀의 오즈비도 「이론 3. 로지스틱 회귀」 양식으로 한 문장 적을 것
> - 견본 3건 요율 표에 차량 모형 쪽 최고값이 최저값의 몇 배인지 한 행을 함께 낼 것
> - 브리핑 3개 항목과 디지털 채널팀 회신을 「PROMPT 셀」 아래 양식에 채울 것
>
> 마감: 2026-10-01T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | 「PROMPT 셀」의 `PROMPT` 에 목표·쓸 변수·출력 형식이 내 말로 적혀 있다 | 25 |
> | 검증 설계 | 값 3개와 그래프가 모두 통과로 열린 상태이고 브리핑 3개 항목이 채워져 있다 | 25 |
> | 오류 탐지 | 로그 스케일의 계수와 사고율비·오즈비를 혼동하지 않고 배율 문장을 적었다 | 25 |
> | 해석·보고 | 채널팀 회신에 결론·근거·한계를 각 한 문장씩 5문장 안에 적었다 | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 첫 화면에 쓸 차량 정보 모형

#### 할 일 — 과제 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시키고, 미션 1 의 3번부터 같은 순서로 진행합니다.
3. 아래 양식에 브리핑 3개 항목과 채널팀 회신을 채워 제출합니다.

**이 과제의 질문** — 차량 정보만 받아 첫 화면 견적을 내도 쓸 만한 숫자가 나옵니까?

오늘 내는 값은 금액이 아니라 연간 사고율입니다. 금액으로 옮기려면 건당 평균 보험금이 더 필요하니, 회신의 한계 문장에 그 점을 함께 적습니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 계약 678,013행 표, 사고 여부 열 `사고여부` 가 더해져 있습니다 | 고정 |
| `CANDIDATES` | 후보 4개의 이름과 formula, 키는 `null` · `vehicle` · `core` · `full` | 고정 |
| `F_VEH` | 차량 정보만 쓰는 후보의 formula 문자열 | 고정 |
| `F_VEH_BIN` | 같은 우변에 타깃만 사고 여부인 formula 문자열 | 고정 |
| `off` | 노출의 로그, 오프셋에 넘길 값 (노출 1년이면 0) | 고정 |
| `SAMPLE3` | 첫 화면 견적을 낼 견본 계약 3건 표 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **차량 모형의 AIC** — 단위 점 · 소수점 아래 2자리 · 후보 4개 채점표에서 `vehicle` 행의 AIC
2. **차량 모형의 차령 1년 사고율비** — 단위 배 · 소수점 아래 4자리 · 차량 모형 계수표의 `VehAge` 한 행을 지수로 변환한 값
3. **견본 계약 세 건 예측 요율의 최대·최소 비** — 단위 배 · 소수점 아래 4자리 · `SAMPLE3` 3건에 차량 모형 예측을 적용해 최고값을 최저값으로 나눈 배수
   - 견본 3건의 비는 예시입니다. 계약 전체의 예측이 어느 범위에 들어오는지는 6번 곡선에서 95분위를 5분위로 나눈 값으로 읽습니다. 곡선의 양 끝인 0분위와 100분위는 계약 한 건이 정하므로 이 비에 넣지 않습니다 (아래 히스토그램 그림)
4. **확인 한 행** — 「네 적합의 관측 수가 모두 같은가 : True」 형식으로 한 행 더 출력합니다
5. **함께 출력할 표 3개와 행 1개** — 후보 4개 채점표(미션 3 과 같은 열) · 차량 모형 계수를 전부 배율로 변환한 표 · 견본 3건에 두 모형 예측 요율을 나란히 놓은 표, 그리고 `F_VEH_BIN` 으로 돌린 **이항 GLM** 의 차령 1년 **오즈비** 한 행
   - 이항 GLM 은 **로지스틱 회귀**와 같은 모형입니다 (이론 3. 로지스틱 회귀) — 타깃이 0·1 이라 분포를 이항으로 바꾼 GLM 이고, 오프셋은 넘기지 않습니다
   - 오즈비는 오즈(사고 날 확률 ÷ 안 날 확률)에 곱해지는 배율입니다
   - 제출물에 그대로 포함되는 항목입니다
6. **그래프 한 장** — 가로축은 백분위 0 ～ 100%, 세로축은 예측 연간 사고율인 꺾은선 그래프
   - 곡선 : 2개. 범례 문구는 「차량 모형」·「핵심 모형」
   - 늘어놓는 순서 : 계약 전체를 예측 요율이 낮은 쪽부터, 백분위는 1% 씩
   - 예측 기준 : 노출 1년 기준 건/년 (계약 행을 넣어 예측하면 오프셋이 0 이 됩니다)
   - 세로축 눈금 : 로그 눈금
   - 가로축 이름 : 「계약을 예측 요율 순으로 늘어놓은 백분위 (%)」
   - 세로축 이름 : 「예측 연간 사고율 (건/년, 노출 1년 기준, 로그 눈금)」

**출력 형식** — 값 3개는 「이름 = 값 단위」 형식으로 한 행씩 `print` 하고, 확인 한 행을 그 아래 함께 출력합니다. 그래프는 한 장입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `CANDIDATES` · `F_VEH_BIN` · `off` · `SAMPLE3` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **같은 조건** — 후보 4개를 한 반복문에서 같은 분포·같은 타깃·같은 오프셋으로 적합하라고 적습니다. 차량 모형만 따로 시키면 그 행에서만 오프셋이 빠집니다.
- **이항 GLM 한 행** — `F_VEH_BIN` 쪽은 분포를 이항으로 명시하고 오프셋은 넘기지 말라고 적습니다. 그 계수에 지수를 씌운 값이 오즈비입니다.
- **출력** — 값 3개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지. 예측 요율은 「노출 1년 기준, 건/년」이라는 것도 함께 적습니다.
- **그래프** — 가로축·세로축에 무엇을 놓을지 · 곡선 2개 · 세로축 눈금 · 가로축·세로축 이름과 범례.

**막히면** — AI 튜터에게 「적합 결과에 계약 표를 넣어 예측하면 오프셋은 어떤 값이 되나요」 처럼 물어보세요.

아래 「PROMPT 셀」은 후보 4개를 적합하느라 10여 초가 걸립니다 — 멈춘 것이 아닙니다.

<details class="lms-extra"><summary>배경 — 채널팀의 의뢰와 사람 정보를 대체할 차량 정보 4개</summary>

미션 3 의 채점표로 보고하려는데 채널팀의 마지막 의뢰가 들어옵니다 — **"온라인 견적 첫 화면에서는 나이나 할증점수를 못 물어봅니다. 가입 전이라 조회 권한이 없어서요. 차량 정보만 받아서 사전 견적을 보여 주려는데, 그 정도로도 쓸 만한 숫자가 나옵니까?"**

새로 배울 방법은 없습니다 — 후보를 하나 더 만들어 같은 조건에서 함께 비교하면 됩니다.

| 컬럼 | 뜻 | 단위·수준 |
|---|---|---|
| `VehPower` | 차량 출력 등급 | 점 |
| `VehAge` | **차령** — 차량의 나이 | 년 |
| `VehGas` | 연료 | 경유 `Diesel` · 휘발유 `Regular` 둘 |
| `VehBrand` | 브랜드 | `B1` 부터 `B14` 까지 이름이 부여된 11개 수준 — 번호가 중간에 비어 14개가 아닙니다 |

비교 조건은 그대로입니다 — 같은 관측 집합 · 같은 타깃 · 같은 오프셋. 달라지는 것은 우변의 특성뿐입니다. 기준선보다 AIC 가 낮으면 차량 정보가 사고율을 설명한다는 뜻이고, 핵심 모형보다 AIC 가 얼마나 높은지가 「사람 정보를 못 물어봐서 잃는 것」의 크기입니다.

</details>

<details class="lms-extra"><summary>배경 — 차령 배율을 읽을 때의 전제 조건</summary>

차령 한 행에 지수를 씌우면 「차령이 1년 증가하면 사고율에 몇 배」라는 요율 문장이 됩니다 (이론 6. 실제 데이터 적용).

그런데 이 배율은 **보정된 효과**입니다. 같이 넣은 브랜드·출력·연료를 고정한 채 읽는 값이라, 브랜드를 제외하고 다시 적합하면 같은 데이터에서도 값이 달라집니다. 그래서 회신에 배율을 적을 때는 **무엇과 함께 적합한 계수인지**를 함께 적습니다 — 「이론 6. 실제 데이터 적용」의 나이 배율이 10년당 0.954 와 1.074 로 달라졌던 사례가 그 증거입니다.

</details>

<details class="lms-extra"><summary>주의 — 오프셋과 예측에서 자주 발생하는 오류</summary>

- 차량 모형만 따로 적합하라고 시키면 오류 메시지 없이 오프셋이 빠져, 채점표의 그 행만 다른 값을 측정하게 됩니다.
- 계약 행을 넣어 `모형.predict(그 행들)` 을 부르면 오프셋이 0 이 되어 **노출 1년 기준 요율**이 나옵니다. `fittedvalues` 는 **노출이 곱해진 건수**라 요율이 아닙니다 — 노출 0.3년이면 요율의 30% 로 출력됩니다.
- 로그 스케일의 계수와 배율을 섞지 않습니다. 사고율비는 사고율에, 이항 GLM 의 오즈비는 오즈에 곱해지는 배율이라 같은 말이 아닙니다 (이론 3. 로지스틱 회귀).
- 이항 GLM 의 오즈비는 오프셋 없이 구한 값입니다. 노출 중앙값이 0.49년이고 계약의 32.9%가 0.25년 이하라 관찰 기간이 짧은 계약이 함께 들어가 있으니, 회신에 인용할 때 그 단서를 함께 적습니다.

</details>

첫 화면에 뜨는 것은 순위가 아니라 계약 한 건의 숫자라, 예측 사고율이 어느 범위에 들어오는지까지 봐야 합니다. 아래 그림이 그 예입니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_002.png" width="640" height="240"/>
<p align="center"><em>▲ 모형마다 예측이 어디까지 분포하는지 비교한 히스토그램 — <strong>Ridge 는 0～0.5, PoissonRegressor 는 0～1.6, HistGradientBoosting 은 0～4</strong> 로, 오른쪽 모형일수록 위험한 계약에 높은 값을 매길 수 있다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>자세히 — 히스토그램 8개를 읽는 법</summary>

위가 학습 표본, 아래가 평가 표본이고 세로축은 계약 수(로그 눈금)입니다. 맨 왼쪽 Data 그래프는 실제 관측된 사고 빈도라 가로축이 0 에서 30 까지 이어집니다. 읽을 것은 각 히스토그램이 오른쪽으로 어디까지 이어지는가 하나입니다.

</details>

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다
if "df" not in globals():             # 맨 위 설치·폰트 셀만 실행해 두면 이 과제만 따로 열어도 여기부터 시작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 첫날 정한 전처리 두 규칙 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

F_VEH = "ClaimNb ~ VehPower + VehAge + C(VehGas) + C(VehBrand)"       # 쓸 변수 1 — 차량 정보만 쓰는 네 번째 후보
CANDIDATES = {                                                       # 쓸 변수 2 — 후보 4개 (이름 -> formula)
    "null": "ClaimNb ~ 1",
    "vehicle": F_VEH,
    "core": "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)",
    "full": ("ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"
             " + C(VehBrand) + C(Region) + np.log(Density)"),
}
off = np.log(df["Exposure"])                  # 쓸 변수 3 — 노출의 로그 (offset 에 넘길 값, 노출 1년이면 0)
df["사고여부"] = (df["ClaimNb"] > 0).astype(int)   # 쓸 변수 4 — 사고가 한 건이라도 있으면 1, 없으면 0
F_VEH_BIN = "사고여부 ~ VehPower + VehAge + C(VehGas) + C(VehBrand)"  # 쓸 변수 5 — 차량 모형과 같은 우변, 타깃만 사고 여부
SAMPLE3 = pd.DataFrame({                                  # 쓸 변수 6 — 첫 화면 견적을 낼 견본 계약 세 건
    "이름":       ["가 · 새 차 · 고출력", "나 · 5년 · 경유", "다 · 15년 · 소형"],
    "VehBrand":   pd.Categorical(["B12", "B1", "B2"], categories=df["VehBrand"].cat.categories),
    "VehAge":     [0, 5, 15],          # 차령 (년)
    "VehPower":   [10, 6, 4],          # 출력 등급 (점)
    "VehGas":     pd.Categorical(["Regular", "Diesel", "Regular"], categories=df["VehGas"].cat.categories),
    "BonusMalus": [100, 60, 50],       # 아래 셋은 핵심 모형 쪽 예측에만 쓰입니다 (점 · 살 · 등급)
    "DrivAge":    [30, 45, 55],
    "Area":       pd.Categorical(["C", "B", "A"], categories=df["Area"].cat.categories),
})

for name, formula in CANDIDATES.items():
    print(f"{name:8s} : {formula}")
print(f"\n차량 정보 넷이 만드는 범주 수준 — VehGas {df['VehGas'].nunique()}개 수준 {sorted(df['VehGas'].unique())}"
      f" · VehBrand {df['VehBrand'].nunique()}개 수준 (기준 하나를 뺀 나머지가 더미로 들어갑니다)")
print(f"계약 {len(df):,} 건 | log(n) = {np.log(len(df)):.4f}"
      "  -> BIC 가 계수 1개마다 더하는 패널티 항 (AIC 는 2)")
print("\n견본 계약 세 건 — 첫 화면에 자주 들어오는 세 유형을 상품팀이 골라 준 것입니다")
print(SAMPLE3.to_string(index=False))
print("쓸 수 있는 이름 :  df · CANDIDATES · F_VEH · F_VEH_BIN (사고 여부용 식) · off"
      " · SAMPLE3 (견본 계약 3건)")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **브리핑 3개 항목과 채널팀 회신** — 채점 결과가 전부 통과로 열리면 이 셀을 채워 그대로 제출합니다.
>
> - **결론** — 첫 화면에 쓸 모형과 그 근거 한 문장 (후보 4개 채점표에서 차량 모형이 어느 위치인지)
> - **위험** — 이 값을 그대로 견적에 쓸 때 문제가 되는 점 한 문장 (6번 곡선의 95분위 ÷ 5분위)
> - **코드가 답을 만드는 방식** — 후보 4개를 같은 조건에서 비교한 부분이 코드의 어느 행인지 한 문장 (같은 관측 집합 · 같은 타깃 · 같은 오프셋)
> - **채널팀 회신 3문장** — 결론 한 문장 · 근거 한 문장 · 한계 한 문장. 여기 적은 3문장을 그대로 회신으로 보냅니다.

## 오늘의 정리 — 계수의 신뢰성 검증 예고

오늘의 질문은 언더라이팅 팀이 던진 「같은 세그먼트 안에서도 사람이 다른데, 계약마다 다른 요율을 어떻게 매기나」였습니다. 답은 **계약 특성 4개를 우변에 넣은 포아송 GLM 의 계수 8개**입니다 — 지수를 씌우면 그대로 요율 배율이 되는 그 표입니다.

채점표에서 1등은 확장 모형이었습니다 — AIC 도 BIC 도 그쪽이 낮았습니다. 그런데도 오늘 요율 문장으로 옮겨 적을 것은 계수 8개인 핵심 모형입니다. 「이론 7. 정리」 끝의 질문대로, 확장 모형의 지역 계수는 계약 1,326건으로 정해진 값이기 때문입니다. 점수는 이 표본 안의 순위일 뿐이라, 후보 선택의 확정은 내일 교차검증 뒤로 넘깁니다.

프라이싱 분석가로서의 셋째 날 보고를 정리합니다. 어제 242개 세그먼트로 나눈 사고율을 오늘은 계약 한 건 한 건에 서로 다른 값으로 매겼습니다. 그 사이를 이은 것이 GLM 의 구성요소 3개, 곧 건수 데이터에 맞는 **포아송 분포**와 계약 특성 4개를 더해 만든 **선형예측자**와 곱셈 관계를 덧셈으로 변환하는 **로그 링크**입니다. 여기에 **오프셋** 항 1개가 추가됩니다 — 계수표에도, 계수 개수 $k$ 에도 포함하지 않는 항입니다.

| 오늘 판단이 필요했던 항목 | 우리가 얻은 답 |
|---|---|
| 어제의 한 숫자를 GLM 이 그대로 돌려주나? | 설명변수를 하나도 안 넣은 null 모형의 $\exp(\text{절편})$ 이 어제의 전체 계약 사고율과 일치했습니다 — GLM 적합이 어제의 최대우도추정을 계수 여러 개로 늘린 것이라는 증거입니다 |
| 계수를 요율표에 어떻게 옮기나? | **로그 스케일에서 더해지던 값이 원래 단위에서는 곱해집니다.** $\exp(\text{계수})$ 가 사고율비이고, 배율 1.0 이 「영향 없음」에 해당하는 값입니다 |
| 계수를 백분율로 바로 읽어도 되나? | 계수가 작을 때만 통하는 근사입니다. 변환 규칙은 계수의 크기와 상관없이 지수 하나뿐이고, 함정 미션에서 값이 달라진 지점이 거기였습니다 |
| 계수 하나를 떼어 인용해도 되나? | 안 됩니다. 나이 배율은 나이만 넣으면 10년당 0.954배인데 할증점수를 함께 넣으면 1.074배로 부호까지 반대가 됩니다(「이론 6. 실제 데이터 적용」, 두 변수의 상관 −0.48). 미션 2 계수표의 `DrivAge` 부호가 그쪽이니, 무엇과 함께 적합한 계수인지를 밝히지 않은 인용은 사실과 다른 문장이 됩니다 |
| 변수를 어디까지 넣나? | 로그우도만으로는 멈출 곳을 못 정해, 계수 개수만큼 패널티 항을 더한 **AIC·BIC** 로 후보의 순위를 매겼습니다 (미션 3) |
| 그 순위는 언제 뜻을 갖나? | 같은 관측 집합 · 같은 타깃 · 같은 오프셋 — 세 조건 위에서만입니다. 하나라도 다르면 로그우도에 더한 항의 수부터 달라져 순위가 뜻을 잃습니다 (미션 3) |

오늘 확정한 것은 2개입니다. 하나는 어제와 같은 **전처리 규칙**이고, 다른 하나는 **후보 채점표**로 미션 3 의 `null` · `core` · `full` 입니다. 과제까지 하면 차량 모형을 더해 4개가 되고, 이 둘이 내일의 출발선입니다.

## 내일 예고 — 이 계수를 믿어도 되는가

오늘 요율 문장으로 옮긴 배율 8개를 경영진 보고서에 그대로 적을 수 있을까요?

「이론 4. 포아송 회귀」 계수표에서 할증점수 계수 옆에는 **표준오차**가 나란히 있었습니다. 그렇다면 "1점당 몇 % 할증"이라고 단정할 게 아니라 "몇 %에서 몇 % 사이"라고 적어야 하는 것 아닐까요.

내일은 이 질문들에 답합니다. **부트스트랩**으로 표본을 다시 추출해 계수가 얼마나 달라지는지 측정하고, **신뢰구간**을 붙여 점추정 하나가 아니라 구간으로 보고합니다. **정규화**(Ridge·Lasso)로 불어난 계수를 안정화하는데, 그것이 어제 배운 MAP 와 같은 방법이었다는 것도 확인합니다.

마지막으로 **교차검증**으로 오늘의 AIC 순위가 새 데이터에서도 유지되는지 봅니다. 미션 3 퀴즈의 주장을 실제로 측정하는 작업입니다. 오늘 미뤄 둔 것이 하나 더 있습니다. 포아송 분포는 분산이 평균과 같다고 가정하는데, 실제 분산이 그보다 크면 계수표의 표준오차가 실제보다 좁게 나옵니다. 그 차이를 측정해 표준오차를 고쳐 읽는 작업은 내일 「이론 5. 잔차 진단」에서 진행합니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/K-fold%20cross%20validation%20EN.svg" width="520" height="258"/>
<p align="center"><em>▲ 내일 「새 데이터에서도 이 순위가 유지되는가」 를 측정하는 방법인 <strong>k겹 교차검증</strong>(k-fold cross-validation) — 오늘 AIC 는 그 성능을 계수 개수로 어림잡기만 했는데, 내일은 이렇게 직접 측정해 오늘의 순위와 대조한다 (출처: Wikimedia Commons, Gufosowa, CC BY-SA 4.0)</em></p>

<details class="lms-extra"><summary>자세히 — k겹 교차검증 1회 반복의 순서</summary>

데이터를 k 조각으로 나눈 뒤 한 번에 한 조각씩 평가용으로 빼고 나머지 전부로 학습하기를 k 번 반복하면, 모든 관측이 정확히 한 번씩 평가용이 되고 그 k 번의 평가 결과를 평균해 「새 데이터에서의 성능」 을 측정합니다.

</details>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_ridge_path_001.png" width="480" height="360"/>
<p align="center"><em>▲ 릿지 정규화의 세기 <code>alpha</code> 를 바꿔 가며 계수 10개가 어떻게 변하는지 그린 그림 — 오른쪽에서 +270 에서 −180 까지 퍼져 있던 선 10개가 왼쪽으로 갈수록 한꺼번에 0 쪽으로 줄어든다 (출처: scikit-learn documentation)</em></p>

가로축이 <code>alpha</code> 인데 눈금이 반대 방향이라 **왼쪽 끝이 강한 정규화** 10⁻² 이고 **오른쪽 끝이 약한 정규화** 10⁻¹⁰ 입니다. 오늘 확장 모형에서 계수 41개로 늘어난 문제를 내일 이 `alpha` 값으로 조정합니다.

오늘 채점이 열어 준 완성본과 내 코드가 달랐던 부분 1곳, 그리고 그 부분에서 무엇을 배웠는지 한 문장 — 2문장이면 충분합니다. 팀 채널에 올려 두고 다른 사람의 2문장을 읽는 것이 오늘의 마지막 복습입니다. 함께 볼 팀이 없다면 내일의 나에게 남기는 메모로 적어 두면 같은 역할을 합니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 모형추정 Day 3 완료</h2>
<div style="color:#656565">분포와 링크를 교체해 계약마다 다른 요율을 계산하고 계수를 요율 문장으로 옮겼습니다.<br/>
내일은 그 값에 신뢰구간을 붙여 보고할 수 있게 만듭니다.<br/>
마지막으로 아래 3문장 퀴즈를 채우면 오늘 분량이 끝납니다.</div>
</div>

> **[문제] 요율 개정 회의 보고용 3문장**
>
> 다음 분기 요율안을 한 장으로 보고하는 상황입니다. 오늘 만든 것만으로 다음 3문장을 채워 보세요. (가) 할증점수를 **요율 문장**으로 옮긴 한 문장 — 무엇이 얼마 오르면 사고율이 몇 배가 되는지, 그리고 그 배율을 계수에서 어떻게 얻었는지 (나) 그 문장에 반드시 함께 적어야 하는 단서 한 문장 — 어떤 변수들과 함께 적합한 계수인지를 왜 밝혀야 하는지 (다) 요율표로 삼을 후보를 어떻게 골랐는지 한 문장 — 무엇을 재서 순위를 매겼고, 그 순위 매기기가 성립하려면 후보들이 무엇을 공유해야 했는지